# 🖼️ easyai-z-image-turbo
> **ComfyUI + Gradio on Google Colab **

---

### What this notebook does

This notebook runs **Z-Image Turbo** — a fast flow-matching image generation model (NextDiT/fp8) — through a full-featured Gradio UI, directly on a Colab

**Run the cells in order (1 → 2 → 2B → 2C → 2D → 2E → 2F → 2G → 3A), then use the Gradio link.**

---

### Tabs available in the UI

| Tab | What it does |
|-----|-------------|
| 🎨 **Generate** | Text-to-image with Standard, Enhanced, Z-Sampler Turbo (BRAVO 3-stage), Flow-DPO lighting |
| 🧠 **Model** | Download & switch diffusion models at runtime |
| 🔧 **LoRA** | Load up to 3 LoRAs with strength control, A/B test, download from URL |
| ⬆️ **Upscale** | SeedVR2 upscaler → Z-Image Refine → Detail Daemon → Auto Color → Final Lanczos |
| 🩹 **Fix** | Auto-fix faces/hands (YOLO + CropAndStitch), Manual inpaint, Pose editor |
| 🎲 **Dual CFG** | Split-step low/high CFG for creative variation |
| ✍️ **Prompt** | QwenVL prompt enhancement and image description *(optional — toggle in Cell 1)* |

---

### Cell 1 options

| Setting |  | Description |
|---------|---------|-------------|
| `CIVITAI_API_KEY` |  | Paste your key for gated Civitai downloads |
| `DOWNLOAD_CONTROLNET_UNION` | | ~2 GB — needed for Pose Edit tab |
| `DOWNLOAD_QWEN_EDIT` | | ~22 GB — needed for Qwen pose transfer |
| `ENABLE_QWENVL_PROMPT` | | QwenVL prompt enhance & image describe |

---

> ⚠️ **Note:** The negative prompt has no effect at CFG 1.0 (the default). It only matters if you raise CFG above 1.0.

In [ ]:
# @title **Cell 1 — Install & Download**
# @markdown ---
# @markdown ### ⚙️ Settings (edit before running)
CIVITAI_API_KEY = "" # @param {type:"string"}
# @markdown > *Optional — paste your Civitai API key for LoRA downloads*
DOWNLOAD_CONTROLNET_UNION = False # @param {type:"boolean"}
# @markdown > *~2 GB — needed for Pose Edit tab*
DOWNLOAD_QWEN_EDIT = False # @param {type:"boolean"}
# @markdown > *~22 GB — Qwen Edit pose transfer (slow)*
ENABLE_QWENVL_PROMPT = False # @param {type:"boolean"}
# @markdown > *QwenVL prompt enhance & image describe (~3 GB on first use)*
# @markdown ---

%cd /content
import os, time

if not os.path.exists("/content/ComfyUI/requirements.txt"):
    os.system("rm -rf /content/ComfyUI")
    for attempt in range(5):
        print(f"🔄 Clone attempt {attempt + 1}/5 ...")
        ret = os.system("git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")
        if ret == 0 and os.path.exists("/content/ComfyUI/requirements.txt"):
            print("✅ Clone successful!")
            break
        else:
            os.system("rm -rf /content/ComfyUI")
            time.sleep(15)
    else:
        raise RuntimeError("Could not clone ComfyUI. Try: Runtime → Disconnect and delete runtime")
else:
    print("✅ ComfyUI already cloned")

%cd /content/ComfyUI

!grep -viE '^(torch|torchvision|torchaudio|torchsde)' requirements.txt > /tmp/req_notorch.txt
!pip install -r /tmp/req_notorch.txt
!pip install torchsde gradio
!apt -y install -qq aria2

# ── Download diffusion model (civitai — aria2c works fine) ──
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    "https://civitai.red/api/download/models/2918747?fileId=2797141&token={CIVITAI_API_KEY}" \
    -d /content/ComfyUI/models/diffusion_models -o z-image-turbo-fp8-e4m3fn.safetensors

# ── Download CLIP & VAE (HuggingFace — wget avoids 403 errors) ──
!wget -c -q --show-progress -O /content/ComfyUI/models/clip/qwen_3_4b.safetensors \
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"

!wget -c -q --show-progress -O /content/ComfyUI/models/vae/ae.safetensors \
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

# ── Download Flow-DPO photorealistic lighting LoRA (~170 MB) ──
FLOW_DPO_PATH = "/content/ComfyUI/models/loras/zit_fdpo_v1.safetensors"
if not os.path.exists(FLOW_DPO_PATH):
    print("⏳ Downloading Flow-DPO photorealistic lighting LoRA...")
    os.system(f'wget -c -q --show-progress -O "{FLOW_DPO_PATH}" '
              f'"https://huggingface.co/F16/z-image-turbo-flow-dpo/resolve/main/zit_fdpo_v1.safetensors"')
    print("✅ Flow-DPO LoRA downloaded")
else:
    print("✅ Flow-DPO LoRA already downloaded")

# ── Clone custom nodes ──
CUSTOM_NODES = "/content/ComfyUI/custom_nodes"

LG_PATH = f"{CUSTOM_NODES}/ComfyUI-LG_SamplingUtils"
if not os.path.exists(f"{LG_PATH}/__init__.py"):
    os.system(f"rm -rf {LG_PATH}")
    os.system(f"git clone https://github.com/LAOGOU-666/ComfyUI-LG_SamplingUtils {LG_PATH}")
    print("✅ LG_SamplingUtils cloned")
else:
    print("✅ LG_SamplingUtils already cloned")

KJ_PATH = f"{CUSTOM_NODES}/ComfyUI-KJNodes"
if not os.path.exists(f"{KJ_PATH}/__init__.py"):
    os.system(f"rm -rf {KJ_PATH}")
    os.system(f"git clone https://github.com/kijai/ComfyUI-KJNodes {KJ_PATH}")
    if os.path.exists(f"{KJ_PATH}/requirements.txt"):
        os.system(f"pip install -q -r {KJ_PATH}/requirements.txt")
    print("✅ KJNodes cloned")
else:
    print("✅ KJNodes already cloned")

V4_PATH = f"{CUSTOM_NODES}/Z-Image-Turbo-Lora-Stack-V4"
if not os.path.exists(f"{V4_PATH}/__init__.py"):
    os.system(f"rm -rf {V4_PATH}")
    os.system(f"git clone https://github.com/aistudynow/Z-Image-Turbo-Lora-Stack-V4 {V4_PATH}")
    print("✅ LoRA Stack V4 cloned")
else:
    print("✅ LoRA Stack V4 already cloned")

ZIT_PATH = f"{CUSTOM_NODES}/ComfyUI-CapitanZiT-Scheduler"
if not os.path.exists(f"{ZIT_PATH}/__init__.py"):
    os.system(f"rm -rf {ZIT_PATH}")
    os.system(f"git clone https://github.com/capitan01R/ComfyUI-CapitanZiT-Scheduler {ZIT_PATH}")
    print("✅ CapitanZiT Scheduler cloned")
else:
    print("✅ CapitanZiT Scheduler already cloned")

FG_PATH = f"{CUSTOM_NODES}/famegrid-auto-color"
if not os.path.exists(f"{FG_PATH}/__init__.py"):
    os.system(f"rm -rf {FG_PATH}")
    os.system(f"git clone https://github.com/ultramuseart/famegrid-auto-color {FG_PATH}")
    print("✅ FameGrid Auto Color cloned")
else:
    print("✅ FameGrid Auto Color already cloned")

ZPOW_PATH = f"{CUSTOM_NODES}/ComfyUI-ZImagePowerNodes"
if not os.path.exists(f"{ZPOW_PATH}/__init__.py"):
    os.system(f"rm -rf {ZPOW_PATH}")
    os.system(f"git clone https://github.com/martin-rizzo/ComfyUI-ZImagePowerNodes {ZPOW_PATH}")
    print("✅ Z-Image Power Nodes cloned")
else:
    print("✅ Z-Image Power Nodes already cloned")

CS_PATH = f"{CUSTOM_NODES}/ComfyUI-Inpaint-CropAndStitch"
if not os.path.exists(f"{CS_PATH}/__init__.py"):
    os.system(f"rm -rf {CS_PATH}")
    os.system(f"git clone https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch {CS_PATH}")
    print("✅ Inpaint CropAndStitch cloned")
else:
    print("✅ Inpaint CropAndStitch already cloned")

# ── ControlNet Aux (DWPose preprocessor for pose-guided relocate) ──
CNAUX_PATH = f"{CUSTOM_NODES}/comfyui_controlnet_aux"
if not os.path.exists(f"{CNAUX_PATH}/__init__.py"):
    os.system(f"rm -rf {CNAUX_PATH}")
    os.system(f"git clone https://github.com/Fannovel16/comfyui_controlnet_aux {CNAUX_PATH}")
    if os.path.exists(f"{CNAUX_PATH}/requirements.txt"):
        os.system(f"pip install -q -r {CNAUX_PATH}/requirements.txt")
    os.system("pip install -q onnxruntime-gpu")
    print("✅ ControlNet Aux cloned")
else:
    print("✅ ControlNet Aux already cloned")

# ── Z-Image ControlNet Union (pose/depth/canny — model patch) ──
PATCH_DIR = "/content/ComfyUI/models/model_patches"
os.makedirs(PATCH_DIR, exist_ok=True)
CN_UNION = f"{PATCH_DIR}/Z-Image-Turbo-Fun-Controlnet-Union-2.1-2602-8steps.safetensors"
if DOWNLOAD_CONTROLNET_UNION:
    if not os.path.exists(CN_UNION) or os.path.getsize(CN_UNION) < 1024 * 1024:
        print("⏳ Downloading Z-Image ControlNet Union 2.1 (~2 GB)...")
        os.system(f'wget -c -q --show-progress -O "{CN_UNION}" '
                  f'"https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union-2.1/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union-2.1-2602-8steps.safetensors"')
        if os.path.exists(CN_UNION) and os.path.getsize(CN_UNION) > 1024 * 1024:
            print(f"✅ ControlNet Union downloaded ({os.path.getsize(CN_UNION)/(1024**3):.1f} GB)")
        else:
            print("❌ ControlNet Union download failed")
    else:
        print("✅ ControlNet Union already downloaded")
else:
    print("⏭️ ControlNet Union skipped (DOWNLOAD_CONTROLNET_UNION=False) — Pose Edit tab won't work")

# ══════════════════════════════════════════════════════════════
# QWEN-IMAGE-EDIT-2511 — real pose transfer (instruction-following editor)
# ~22GB total download. Set to False to skip if you don't need pose transfer.
# ══════════════════════════════════════════════════════════════

if DOWNLOAD_QWEN_EDIT:
    # GGUF loader node
    GGUF_PATH = f"{CUSTOM_NODES}/ComfyUI-GGUF"
    if not os.path.exists(f"{GGUF_PATH}/__init__.py"):
        os.system(f"rm -rf {GGUF_PATH}")
        os.system(f"git clone https://github.com/city96/ComfyUI-GGUF {GGUF_PATH}")
        os.system(f"pip install -q -r {GGUF_PATH}/requirements.txt")
        print("✅ ComfyUI-GGUF cloned")
    else:
        print("✅ ComfyUI-GGUF already cloned")

    # Qwen Edit 2511 Q4_K_M (13.1GB)
    QE_MODEL = "/content/ComfyUI/models/diffusion_models/qwen-image-edit-2511-Q4_K_M.gguf"
    if not os.path.exists(QE_MODEL) or os.path.getsize(QE_MODEL) < 10 * 1024**3:
        print("⏳ Downloading Qwen-Image-Edit-2511 Q4_K_M (13.1GB — this takes a while)...")
        os.system(f'wget -c -q --show-progress -O "{QE_MODEL}" '
                  f'"https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q4_K_M.gguf"')
        print(f"✅ Qwen Edit model ({os.path.getsize(QE_MODEL)/(1024**3):.1f} GB)" if os.path.exists(QE_MODEL) else "❌ Qwen Edit download failed")
    else:
        print("✅ Qwen Edit model already downloaded")

    # Qwen 2.5 VL 7B text encoder (fp8, ~8GB)
    QE_CLIP = "/content/ComfyUI/models/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors"
    os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
    if not os.path.exists(QE_CLIP) or os.path.getsize(QE_CLIP) < 5 * 1024**3:
        print("⏳ Downloading Qwen 2.5 VL 7B encoder (8GB)...")
        os.system(f'wget -c -q --show-progress -O "{QE_CLIP}" '
                  f'"https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors"')
        print("✅ Qwen VL encoder downloaded" if os.path.exists(QE_CLIP) else "❌ Encoder download failed")
    else:
        print("✅ Qwen VL encoder already downloaded")

    # Qwen Image VAE
    QE_VAE = "/content/ComfyUI/models/vae/qwen_image_vae.safetensors"
    if not os.path.exists(QE_VAE):
        print("⏳ Downloading Qwen Image VAE...")
        os.system(f'wget -c -q --show-progress -O "{QE_VAE}" '
                  f'"https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors"')
        print("✅ Qwen VAE downloaded" if os.path.exists(QE_VAE) else "❌ VAE download failed")
    else:
        print("✅ Qwen VAE already downloaded")

    # Lightning 4-step LoRA (makes it fast)
    QE_LORA = "/content/ComfyUI/models/loras/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"
    if not os.path.exists(QE_LORA):
        print("⏳ Downloading Qwen Edit Lightning 4-step LoRA...")
        os.system(f'wget -c -q --show-progress -O "{QE_LORA}" '
                  f'"https://huggingface.co/lightx2v/Qwen-Image-Edit-2511-Lightning/resolve/main/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"')
        print("✅ Lightning LoRA downloaded" if os.path.exists(QE_LORA) else "❌ Lightning LoRA download failed")
    else:
        print("✅ Lightning LoRA already downloaded")
else:
    print("⏭️ Qwen Edit download skipped (DOWNLOAD_QWEN_EDIT=False)")

print("\n🎉 All ready!")

In [ ]:
# @title Cell 2 — Load Models & Generate

# ════════════════════════════════════════════════════════════════════
# CELL 2 — LOAD Z-IMAGE MODELS & DEFINE GENERATION FUNCTIONS
# V4 LoRA stack, Flow-DPO, civitai, composition (prompt-based)
# ANC, Z-Sampler Turbo (BRAVO 3-stage pipeline)
# Run after Cell 1
# ════════════════════════════════════════════════════════════════════

%cd /content/ComfyUI

import os, sys, glob, random, time, re, subprocess, urllib.parse
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import gradio as gr

import comfy.sample
import comfy.samplers
import comfy.sampler_helpers
import comfy.utils

from nodes import NODE_CLASS_MAPPINGS

UNETLoader       = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader       = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader        = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode   = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler         = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode        = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
LoraLoader       = NODE_CLASS_MAPPINGS["LoraLoader"]()

LatentUpscaleBy = NODE_CLASS_MAPPINGS.get("LatentUpscaleBy")
if LatentUpscaleBy:
    LatentUpscaleBy = LatentUpscaleBy()
    print("✅ LatentUpscaleBy loaded")

DIFFUSION_DIR = "/content/ComfyUI/models/diffusion_models"
CLIP_DIR      = "/content/ComfyUI/models/clip"
VAE_DIR       = "/content/ComfyUI/models/vae"

print("⏳ Loading base models into VRAM ...")
with torch.inference_mode():
    unet_base = UNETLoader.load_unet("z-image-turbo-fp8-e4m3fn.safetensors", "fp8_e4m3fn_fast")[0]
    clip_base = CLIPLoader.load_clip("qwen_3_4b.safetensors", type="lumina2")[0]
    vae       = VAELoader.load_vae("ae.safetensors")[0]

print("✅ All models loaded!")

OUTPUT_DIR  = "/content/ComfyUI/output"
LORA_DIR    = "/content/ComfyUI/models/loras"
LORA_FOLDER = LORA_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
print(f"✅ LoRA folder: {LORA_FOLDER}")

current_model_name  = "z-image-turbo-fp8-e4m3fn.safetensors"
current_model_dtype = "fp8_e4m3fn_fast"

CIVITAI_TOKEN = CIVITAI_API_KEY  # From Cell 1 form
def _append_civitai_token(url):
    sep = "&" if "?" in url else "?"
    return f"{url}{sep}token={CIVITAI_TOKEN}"

stop_flag = False
def request_stop():
    global stop_flag; stop_flag = True
    return "⏹️ Stop requested."
def reset_stop():
    global stop_flag; stop_flag = False

active_unet = unet_base
active_clip = clip_base
loaded_loras_info = ""

def _symlink_lora(name):
    name = name.strip()
    link = os.path.join(LORA_DIR, name)
    if os.path.exists(link): return name
    src = os.path.join(LORA_FOLDER, name)
    if os.path.exists(src): os.symlink(src, link); return name
    raise FileNotFoundError(f"LoRA not found: {name}")

# ── LORA ENGINE (standard patching with diagnostics) ──
# Uses ComfyUI's standard add_patches system which stores deltas as metadata.
# Patches are applied during forward pass via calculate_weight().
# Stacks cleanly — no forward hook conflicts.
import comfy.lora, comfy.lora_convert
import folder_paths, hashlib

_LORA_SD_CACHE = {}

def _lora_path_for(name):
    try: return folder_paths.get_full_path_or_raise("loras", name)
    except Exception: pass
    for d in (LORA_DIR, LORA_FOLDER):
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return None

def _load_lora_sd(name):
    path = _lora_path_for(name)
    if not path: raise FileNotFoundError(f"LoRA not found: {name}")
    if path not in _LORA_SD_CACHE:
        _LORA_SD_CACHE[path] = comfy.utils.load_torch_file(path, safe_load=True)
    return _LORA_SD_CACHE[path]

def _analyze_lora_keys(lora_sd):
    keys = list(lora_sd.keys())
    fmt = "unknown"
    if any(".lora_A." in k or ".lora_B." in k for k in keys): fmt = "lora_A/B (ai-toolkit/diffusers)"
    elif any(".lora_up." in k or ".lora_down." in k for k in keys): fmt = "lora_up/down (kohya)"
    elif any(".diff" in k for k in keys): fmt = "diff (full delta)"
    prefixes = set()
    for k in keys[:200]:
        parts = k.split(".")
        if len(parts) >= 2: prefixes.add(parts[0])
    return fmt, sorted(prefixes)[:6]

def _apply_lora_standard(model, clip, lora_name, strength_model, strength_clip=0.0, tag=None):
    """
    Standard patching LoRA loader with diagnostics.
    Returns (model, clip, info).
    """
    info = dict(name=lora_name, matched=0, unmatched=0, total_keys=0,
                format="?", prefixes=[], ok=False, error=None)
    if strength_model == 0 and strength_clip == 0:
        info["error"] = "strength is 0"; return model, clip, info
    try:
        lora_sd = _load_lora_sd(lora_name)
    except Exception as e:
        info["error"] = str(e); return model, clip, info

    info["total_keys"] = len(lora_sd)
    info["format"], info["prefixes"] = _analyze_lora_keys(lora_sd)

    # Build key map
    key_map = {}
    if model is not None:
        key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
    if clip is not None and strength_clip != 0:
        key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)

    # Convert + load
    converted = comfy.lora_convert.convert_lora(lora_sd)
    loaded = comfy.lora.load_lora(converted, key_map, log_missing=False)

    if not loaded:
        info["error"] = f"0/{info['total_keys']} keys matched model (wrong architecture?)"
        return model, clip, info

    # Apply via standard patching
    new_model = model
    k_model = set()
    if model is not None:
        new_model = model.clone()
        k_model = set(new_model.add_patches(loaded, strength_model))

    new_clip = clip
    k_clip = set()
    if clip is not None and strength_clip != 0:
        new_clip = clip.clone()
        k_clip = set(new_clip.add_patches(loaded, strength_clip))

    # Count matched vs unmatched
    all_matched = k_model | k_clip
    unmatched = [x for x in loaded if x not in all_matched]
    info["matched"] = len(all_matched)
    info["unmatched"] = len(unmatched)
    info["ok"] = len(all_matched) > 0

    if not info["ok"]:
        info["error"] = f"loaded but 0 patches applied (keys converted but didn't match model weights)"

    return new_model, new_clip, info

def _lora_info_str(info):
    if info["error"] and not info["ok"]:
        return f"❌ {info['name']}: {info['error']} [{info['format']}]"
    return f"✅ {info['name']}: {info['matched']} patches applied" + (f" ({info['unmatched']} skipped)" if info.get('unmatched') else "") + f" [{info['format']}]"

@torch.inference_mode()
def load_loras(sel1, mstr1, en1, sel2, mstr2, en2, sel3, mstr3, en3,
               auto_scale, apply_clip, clip_ratio):
    global active_unet, active_clip, loaded_loras_info
    unet = unet_base; clip = clip_base; log = []
    loras = [("LoRA 1", sel1, mstr1, en1), ("LoRA 2", sel2, mstr2, en2), ("LoRA 3", sel3, mstr3, en3)]
    active_loras = [(n, s, ms, e) for n, s, ms, e in loras
                    if e and s and s != "(no LoRAs found)" and ms > 0]
    num_active = len(active_loras)
    for idx, (name, sel, m_strength, enabled) in enumerate(active_loras):
        try:
            direct_path = os.path.join(LORA_DIR, sel)
            path = direct_path if os.path.exists(direct_path) and not os.path.islink(direct_path) else os.path.join(LORA_FOLDER, sel)
            filename = _symlink_lora(sel)
            m_final = float(m_strength) / num_active if auto_scale and num_active > 1 else float(m_strength)
            c_final = m_final * float(clip_ratio) if apply_clip else 0.0
            unet, clip, info = _apply_lora_standard(unet, clip, filename, m_final, c_final, tag=f"user{idx}")
            size_mb = os.path.getsize(path) / (1024**2)
            scaled_note = f" (÷{num_active})" if auto_scale and num_active > 1 else ""
            clip_note = f" CLIP:{c_final:.2f}" if c_final > 0 else " CLIP:off"
            if info["ok"]:
                log.append(f"✅ {name}: {filename}\n   Model:{m_final:.2f}{clip_note} | {size_mb:.0f}MB{scaled_note}\n   🔌 {info['matched']} patches applied [{info['format']}]")
            else:
                log.append(f"❌ {name}: {filename}\n   {info['error']}\n   keys:{info['total_keys']} fmt:{info['format']} prefixes:{info['prefixes']}")
        except Exception as e: log.append(f"❌ {name}: {str(e)}")
    active_unet = unet; active_clip = clip
    if not log:
        loaded_loras_info = "ℹ️ No LoRAs active."
    else:
        loaded_loras_info = "\n".join(log)
        clip_mode = f"CLIP ratio: {float(clip_ratio):.1f}×" if apply_clip else "CLIP: OFF (V4 stack)"
        loaded_loras_info += f"\n\n📊 {clip_mode}"
        if apply_clip: loaded_loras_info += "\n⚠️ CLIP is ON — if washed out, turn it off."
    return loaded_loras_info

def unload_all_loras():
    global active_unet, active_clip, loaded_loras_info
    active_unet = unet_base; active_clip = clip_base
    loaded_loras_info = "✅ All LoRAs unloaded."
    return loaded_loras_info

def _tensor_to_pil(tensor, index=0):
    return Image.fromarray(np.array(tensor[index].cpu() * 255, dtype=np.uint8))

def _call_node(node_instance, **kwargs):
    fn_name = node_instance.FUNCTION if hasattr(node_instance, 'FUNCTION') else type(node_instance).FUNCTION
    return getattr(node_instance, fn_name)(**kwargs)

last_generated_pil = None
def _get_last_generated():
    global last_generated_pil; return last_generated_pil

WEIGHT_DTYPES = ["fp8_e4m3fn_fast", "fp8_e4m3fn", "fp8_e5m2", "default"]

def scan_diffusion_models():
    return sorted([os.path.basename(f) for f in glob.glob(os.path.join(DIFFUSION_DIR, "*.safetensors"))]) or ["(no models found)"]

def _sanitize_model_filename(name):
    name = re.sub(r'[^\w\-. ()]+', '_', name.strip())
    if not name.lower().endswith('.safetensors'): name = (name.rsplit('.', 1)[0] if '.' in name else name) + '.safetensors'
    return name

def _convert_model_url(url):
    url = url.strip()
    if 'huggingface.co' in url and '/blob/' in url: url = url.replace('/blob/', '/resolve/')
    m = re.match(r'https?://civitai\.com/models/(\d+)(?:/[^?]*)?(?:\?modelVersionId=(\d+))?', url)
    if m:
        if m.group(2): url = f"https://civitai.com/api/download/models/{m.group(2)}"
        else: return None, "❌ Civitai: need the download link."
    return url, None

def download_model(url, custom_name, is_civitai=False):
    if not url or not url.strip(): return "❌ Paste a URL.", gr.update()
    direct_url, error = _convert_model_url(url.strip())
    if error: return error, gr.update()
    if is_civitai: direct_url = _append_civitai_token(direct_url)
    guessed = os.path.basename(urllib.parse.unquote(urllib.parse.urlparse(direct_url.split("?")[0]).path))
    filename = _sanitize_model_filename(custom_name) if custom_name and custom_name.strip() else (
        _sanitize_model_filename(guessed) if guessed and '.' in guessed else f"model_{int(time.time())}.safetensors")
    dest = os.path.join(DIFFUSION_DIR, filename)
    if os.path.exists(dest):
        return f"⚠️ Exists: {filename} ({os.path.getsize(dest)/(1024**2):.0f} MB)", gr.update(choices=scan_diffusion_models(), value=filename)
    safe_url = direct_url.replace('"', '\\"')
    ret = os.system(f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{safe_url}" -d "{DIFFUSION_DIR}" -o "{filename}"')
    if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < 10240:
        if os.path.exists(dest): os.remove(dest)
        os.system(f'wget -c -q --show-progress -O "{dest}" "{safe_url}"')
    if not os.path.exists(dest) or os.path.getsize(dest) < 10240:
        if os.path.exists(dest): os.remove(dest)
        return "❌ Download failed.", gr.update()
    return f"✅ {filename} ({os.path.getsize(dest)/(1024**2):.0f} MB)", gr.update(choices=scan_diffusion_models(), value=filename)

@torch.inference_mode()
def switch_model(model_name, weight_dtype):
    global unet_base, active_unet, active_clip, current_model_name, current_model_dtype
    if not model_name or model_name == "(no models found)": return f"❌ No model selected.\nActive: {current_model_name}"
    if model_name == current_model_name and weight_dtype == current_model_dtype: return f"ℹ️ Already loaded: {current_model_name}"
    try:
        torch.cuda.empty_cache()
        unet_base = UNETLoader.load_unet(model_name, weight_dtype)[0]
        active_unet = unet_base; active_clip = clip_base
        current_model_name = model_name; current_model_dtype = weight_dtype
        torch.cuda.empty_cache()
        return f"✅ Loaded: {model_name}\nType: {weight_dtype}\nℹ️ LoRAs reset."
    except Exception as e:
        torch.cuda.empty_cache(); return f"❌ Failed: {str(e)[:150]}\nActive: {current_model_name}"

def delete_model(model_name):
    if not model_name or model_name in ("(no models found)", current_model_name, "z-image-turbo-fp8-e4m3fn.safetensors"):
        return "❌ Can't delete.", gr.update()
    path = os.path.join(DIFFUSION_DIR, model_name)
    if os.path.exists(path): os.remove(path); return f"✅ Deleted: {model_name}", gr.update(choices=scan_diffusion_models())
    return "❌ Not found.", gr.update()

# ── Flow-DPO ──
FLOW_DPO_LORA = "zit_fdpo_v1.safetensors"
def _apply_flow_dpo(unet, clip, strength=0.8):
    if not os.path.exists(os.path.join(LORA_DIR, FLOW_DPO_LORA)): return unet, clip
    modified_unet, _, info = _apply_lora_standard(unet, clip, FLOW_DPO_LORA, float(strength), 0.0, tag="flowdpo")
    print(_lora_info_str(info))
    return modified_unet, clip



# ── Composition (prompt-based) ──
_COMP_PROMPTS = {
    "rule of thirds (left)": "cinematic composition, subject positioned on the left third of the frame, negative space on the right, rule of thirds framing, ",
    "rule of thirds (right)": "cinematic composition, subject positioned on the right third of the frame, negative space on the left, rule of thirds framing, ",
    "rule of thirds (bottom-L)": "cinematic composition, subject in the lower-left area, open sky or space above, grounded framing, ",
    "rule of thirds (bottom-R)": "cinematic composition, subject in the lower-right area, open space above and to the left, ",
    "centered subject": "symmetrical centered composition, subject perfectly centered in frame, balanced framing, direct eye-level perspective, ",
    "golden spiral": "golden ratio composition, fibonacci spiral framing, organic natural flow, artistic asymmetric balance, ",
    "diagonal (TL→BR)": "dynamic diagonal composition, energy flowing from top-left to bottom-right, dramatic angular framing, ",
    "diagonal (BL→TR)": "dynamic diagonal composition, energy flowing from bottom-left to top-right, ascending diagonal, ",
    "low horizon (big sky)": "low horizon line, expansive dramatic sky taking two-thirds of frame, wide landscape vista, ",
    "high horizon (big ground)": "high horizon line, detailed foreground and ground taking two-thirds of frame, grounded perspective, ",
    "vignette focus": "vignette composition, strong center focus with darker edges, cinematic framing, ",
    "leading lines": "leading lines composition, perspective lines converging toward subject, vanishing point depth, ",
    "light from top-left": "dramatic top-left lighting, warm key light from upper left, natural shadow fall to lower right, ",
    "light from top-right": "dramatic top-right lighting, key light from upper right, natural shadow fall to lower left, ",
    "light from above": "overhead lighting, top-down illumination, shadows falling downward, ",
}
def _apply_composition_to_prompt(prompt, comp_type):
    return _COMP_PROMPTS.get(comp_type, "") + prompt

# ── Enhanced generate placeholders (set in Cell 2C) ──
_ModelSamplingZImage = None
_LGNoiseInjectionLatent = None
_GenerateNoise = None

# ── ANC helpers ──
ADVERSARIAL_REFINE_DENOISE = 0.30
ADVERSARIAL_BLEND_STRENGTH = 0.40

# ── BRAVO sigma preset ──
BRAVO_SIGMA_PRESET = (
    ((0.991, 0.920), (0.942, 0.000), (0.710, 0.000)),
    ((0.991, 0.920), (0.935, 0.789, 0.000), (0.710, 0.000)),
    ((0.991, 0.920), (0.935, 0.789, 0.000), (0.658, 0.302, 0.000)),
    ((0.991, 0.920), (0.935, 0.770, 0.690, 0.000), (0.658, 0.302, 0.000)),
    ((0.991, 0.920), (0.935, 0.900, 0.875, 0.800, 0.000), (0.658, 0.302, 0.000)),
    ((0.991, 0.920), (0.935, 0.900, 0.875, 0.820, 0.750, 0.000), (0.658, 0.302, 0.000)),
    ((0.991, 0.920), (0.935, 0.900, 0.875, 0.820, 0.750, 0.000), (0.658, 0.4556, 0.200, 0.000)),
)

def _get_bravo_sigmas(steps):
    idx = min(max(3, steps), 9) - 3
    preset = BRAVO_SIGMA_PRESET[idx]
    s1 = torch.tensor(preset[0], device='cpu')
    s2 = torch.tensor(preset[1], device='cpu')
    s3 = torch.tensor(preset[2], device='cpu') if preset[2] else None
    return s1, s2, s3

def _sample_stage(latents, model, positive, negative, sigmas, seed,
                  noise_scale=1.0, noise_bias=0.0, add_noise=True,
                  force_final_denoise=True, sampler_name="euler"):
    sampler = comfy.samplers.sampler_object(sampler_name)
    latents = comfy.sample.fix_empty_latent_channels(model, latents)
    if force_final_denoise and sigmas[-1] != 0:
        sigmas = sigmas.clone(); sigmas[-1] = 0
    if add_noise and noise_scale != 0:
        gen = torch.Generator(device="cpu").manual_seed(seed & 0xFFFFFFFFFFFFFFFF)
        noise = torch.randn(latents.shape, generator=gen, dtype=latents.dtype,
                            layout=latents.layout, device="cpu")
        if noise_scale != 1.0: noise = noise * noise_scale
        if isinstance(noise_bias, (int, float)) and noise_bias != 0: noise = noise + noise_bias
        elif isinstance(noise_bias, torch.Tensor): noise = noise + noise_bias
    else:
        noise = torch.zeros(latents.shape, dtype=latents.dtype, layout=latents.layout, device="cpu")
    return comfy.sample.sample_custom(model, noise, 1.0, sampler, sigmas, positive, negative,
                                      latents, noise_mask=None, callback=None,
                                      disable_pbar=True, seed=seed)

# ════════════════════════════════════════════════════════════════════
# GENERATE FUNCTIONS
# ════════════════════════════════════════════════════════════════════

@torch.inference_mode()
def _generate_standard(positive_prompt, negative_prompt, width, height,
                       batch_size, seed, steps, cfg, sampler_name, scheduler, denoise,
                       flow_dpo_enabled=False, flow_dpo_strength=0.8):
    reset_stop()
    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    gen_unet = active_unet; gen_clip = active_clip
    if flow_dpo_enabled:
        yield None, int(seed), "⏳ Applying Flow-DPO..."
        gen_unet, gen_clip = _apply_flow_dpo(active_unet, active_clip, flow_dpo_strength)

    positive = CLIPTextEncode.encode(gen_clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(gen_clip, negative_prompt)[0]
    latent_image = EmptyLatentImage.generate(int(width), int(height), batch_size=int(batch_size))[0]

    yield None, int(seed), "⏳ Generating..."
    samples = KSampler.sample(gen_unet, int(seed), int(steps), float(cfg),
        sampler_name, scheduler, positive, negative, latent_image, denoise=float(denoise))[0]
    decoded = VAEDecode.decode(vae, samples)[0].detach()
    images = []; ts = int(time.time())
    for i in range(decoded.shape[0]):
        img = _tensor_to_pil(decoded, i)
        img.save(f"{OUTPUT_DIR}/z_image_turbo_{ts}_{i}.png"); images.append(img)
    yield images, int(seed), "✅ Done" + (" (Flow-DPO)" if flow_dpo_enabled else "")

@torch.inference_mode()
def _generate_enhanced(positive_prompt, negative_prompt, width, height,
                       batch_size, seed, steps, cfg, sampler_name, scheduler, denoise,
                       flow_dpo_enabled=False, flow_dpo_strength=0.8):
    reset_stop()
    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)
    if not all([_ModelSamplingZImage, _LGNoiseInjectionLatent, _GenerateNoise, LatentUpscaleBy]):
        yield None, int(seed), "❌ Enhanced mode requires Cell 2C."; return

    gen_unet = active_unet; gen_clip = active_clip
    if flow_dpo_enabled:
        yield None, int(seed), "⏳ Applying Flow-DPO..."
        gen_unet, gen_clip = _apply_flow_dpo(active_unet, active_clip, flow_dpo_strength)

    yield None, int(seed), "⏳ Enhanced: preparing..."
    model_zimage = _call_node(_ModelSamplingZImage, model=gen_unet, shift=3, multiplier=1)[0]
    noise_latent = _call_node(_GenerateNoise,
        width=int(width), height=int(height), batch_size=int(batch_size),
        seed=int(seed), multiplier=1.0, constant_batch_noise=False, normalize=False,
        latent_channels="16", shape="BCHW", model=model_zimage, sigmas=None)[0]
    positive = CLIPTextEncode.encode(gen_clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(gen_clip, negative_prompt)[0]

    model_injected = _call_node(_LGNoiseInjectionLatent, model=model_zimage,
        reference_latent=noise_latent, strength=0.4, start_percent=0.0, end_percent=0.2)[0]
    yield None, int(seed), "⏳ Enhanced pass 1/2..."
    pass1 = KSampler.sample(model_injected, int(seed), 6, 1.0, "euler", "simple", positive, negative, noise_latent, denoise=1.0)[0]
    upscaled = LatentUpscaleBy.upscale(pass1, "nearest-exact", 1.0)[0]
    yield None, int(seed), "⏳ Enhanced pass 2/2..."
    try: pass2 = KSampler.sample(model_injected, int(seed)+1, 3, 1.0, "res_multistep", "simple", positive, negative, upscaled, denoise=0.6)[0]
    except: pass2 = KSampler.sample(model_injected, int(seed)+1, 3, 1.0, "dpmpp_2m_sde", "simple", positive, negative, upscaled, denoise=0.6)[0]
    decoded = VAEDecode.decode(vae, pass2)[0].detach()
    images = []; ts = int(time.time())
    for i in range(decoded.shape[0]):
        img = _tensor_to_pil(decoded, i); img.save(f"{OUTPUT_DIR}/z_enhanced_{ts}_{i}.png"); images.append(img)
    yield images, int(seed), "✅ Done (Enhanced)" + (" + Flow-DPO" if flow_dpo_enabled else "")

@torch.inference_mode()
def _generate_zsampler(positive_prompt, negative_prompt, width, height,
                       batch_size, seed, steps, cfg, sampler_name, scheduler, denoise,
                       flow_dpo_enabled=False, flow_dpo_strength=0.8,
                       zs_intensity=0.5, zs_ibias=0.0):
    reset_stop()
    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    gen_unet = active_unet; gen_clip = active_clip
    if flow_dpo_enabled:
        yield None, int(seed), "⏳ Applying Flow-DPO..."
        gen_unet, gen_clip = _apply_flow_dpo(active_unet, active_clip, flow_dpo_strength)

    positive = CLIPTextEncode.encode(gen_clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(gen_clip, negative_prompt)[0]
    latent = EmptyLatentImage.generate(int(width), int(height), batch_size=int(batch_size))[0]
    latents = latent["samples"]

    noise_overdose = float(zs_intensity) * 0.4
    noise_scale = 1.0 + noise_overdose
    bias_level = (float(zs_intensity) + 1) * 4 - 1
    bias_level = min(max(bias_level, 0.0), 4.0)
    bias_level += 10 * float(zs_ibias)
    bias_level = min(max(bias_level, -6.0), 14.0)

    s1, s2, s3 = _get_bravo_sigmas(int(steps))
    total_steps = (len(s1) - 1) + (len(s2) - 1) + (len(s3) - 1 if s3 is not None else 0)

    step = 0
    yield None, int(seed), f"⏳ Z-Sampler [{step+1}/{total_steps}] Stage 1: structure..."
    latents = _sample_stage(latents, gen_unet, positive, negative, s1, int(seed),
                            noise_scale=noise_scale, add_noise=True, force_final_denoise=True)
    step += len(s1) - 1
    if stop_flag: yield None, int(seed), "⏹️ Stopped"; return

    yield None, int(seed), f"⏳ Z-Sampler [{step+1}/{total_steps}] Stage 2: detail..."
    latents = _sample_stage(latents, gen_unet, positive, negative, s2, int(seed)+16,
                            noise_scale=1.0, add_noise=True, force_final_denoise=True)
    step += len(s2) - 1
    if stop_flag: yield None, int(seed), "⏹️ Stopped"; return

    if s3 is not None:
        yield None, int(seed), f"⏳ Z-Sampler [{step+1}/{total_steps}] Stage 3: refine..."
        latents = _sample_stage(latents, gen_unet, positive, negative, s3, 696969,
                                noise_scale=1.0, add_noise=True, force_final_denoise=True)

    decoded = VAEDecode.decode(vae, {"samples": latents})[0].detach()
    images = []; ts = int(time.time())
    for i in range(decoded.shape[0]):
        img = _tensor_to_pil(decoded, i)
        img.save(f"{OUTPUT_DIR}/z_zsampler_{ts}_{i}.png"); images.append(img)
    yield images, int(seed), f"✅ Done (Z-Sampler {int(steps)}s)" + (" + Flow-DPO" if flow_dpo_enabled else "")

# ── LoRA A/B Test ──
@torch.inference_mode()
def lora_ab_test(lora_name, strength, test_prompt, test_seed, test_steps):
    """Same seed, with vs without LoRA. Returns [img_a, img_b], status."""
    if not lora_name or lora_name == "(no LoRAs found)":
        return None, "❌ Select a LoRA."
    if int(test_seed) == 0:
        test_seed = random.randint(1, 2**31)
    prompt = (test_prompt or "").strip() or "photo of a person standing outdoors, natural light, detailed"
    positive = CLIPTextEncode.encode(clip_base, prompt)[0]
    negative = CLIPTextEncode.encode(clip_base, "blurry ugly bad")[0]
    latent = EmptyLatentImage.generate(768, 768, batch_size=1)[0]
    steps = int(test_steps)

    samples_a = KSampler.sample(unet_base, int(test_seed), steps, 1.0,
        "euler", "simple", positive, negative, latent, denoise=1.0)[0]
    img_a = _tensor_to_pil(VAEDecode.decode(vae, samples_a)[0].detach(), 0)

    try: filename = _symlink_lora(lora_name)
    except Exception as e: return [img_a], f"❌ {e}"
    lora_unet, _, info = _apply_lora_standard(unet_base, clip_base, filename, float(strength), 0.0, tag="abtest")
    if not info["ok"]:
        return [img_a], f"❌ {info['error']} | keys:{info['total_keys']} fmt:{info['format']} prefixes:{info['prefixes']}"

    samples_b = KSampler.sample(lora_unet, int(test_seed), steps, 1.0,
        "euler", "simple", positive, negative, latent, denoise=1.0)[0]
    img_b = _tensor_to_pil(VAEDecode.decode(vae, samples_b)[0].detach(), 0)

    diff = float(np.abs(np.array(img_a, dtype=np.float32) - np.array(img_b, dtype=np.float32)).mean())
    la = _stamp_label(img_a, "A — No LoRA")
    lb = _stamp_label(img_b, f"B — LoRA {float(strength):.2f}")
    ts = int(time.time())
    la.save(f"{OUTPUT_DIR}/abtest_a_{ts}.png"); lb.save(f"{OUTPUT_DIR}/abtest_b_{ts}.png")

    verdict = "✅ LoRA is working" if diff >= 4.0 else ("⚠️ Weak effect — try higher strength" if diff >= 1.0 else "❌ No visible effect")
    status = f"{verdict} | pixel diff {diff:.1f}/255 | {info['matched']} patches [{info['format']}] | seed {int(test_seed)}"
    return [la, lb], status

# ── Unified generate ──
def generate(positive_prompt, negative_prompt, width, height,
             batch_size, seed, steps, cfg, sampler_name, scheduler, denoise,
             enhanced=False, flow_dpo_enabled=False, flow_dpo_strength=0.8,
             zsampler_enabled=False, zs_intensity=0.5, zs_ibias=0.0):
    global last_generated_pil

    if zsampler_enabled: fn = _generate_zsampler
    elif enhanced: fn = _generate_enhanced
    else: fn = _generate_standard

    kwargs = dict(
        flow_dpo_enabled=flow_dpo_enabled, flow_dpo_strength=flow_dpo_strength,
    )
    if zsampler_enabled:
        kwargs["zs_intensity"] = zs_intensity
        kwargs["zs_ibias"] = zs_ibias

    for out in fn(positive_prompt, negative_prompt, width, height,
                  batch_size, seed, steps, cfg, sampler_name, scheduler, denoise,
                  **kwargs):
        if out[0] and isinstance(out[0], list) and len(out[0]) > 0:
            last_generated_pil = out[0][0]
        yield out

In [ ]:
# @title Cell 2B — SeedVR2 Upscaler

# ════════════════════════════════════════════════════════════════════
# CELL 2B — INSTALL & LOAD SEEDVR2 UPSCALER
# Pipeline: SeedVR2 upscale → Z-Image refine (optional LoRA)
#         → FameGrid Auto Color (optional) → SeedVR2 final
# Stores before/after for comparison slider
# ════════════════════════════════════════════════════════════════════

%cd /content/ComfyUI

import os, sys, time, asyncio, inspect, threading
import torch
import numpy as np
from PIL import Image
from unittest.mock import MagicMock

SEEDVR2_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler"

if not os.path.exists(os.path.join(SEEDVR2_PATH, "__init__.py")):
    os.system(f"rm -rf {SEEDVR2_PATH}")
    for attempt in range(5):
        print(f"🔄 Cloning SeedVR2 attempt {attempt + 1}/5 ...")
        ret = os.system(f"git clone https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler {SEEDVR2_PATH}")
        if ret == 0 and os.path.exists(os.path.join(SEEDVR2_PATH, "__init__.py")):
            print("✅ SeedVR2 clone successful!")
            break
        os.system(f"rm -rf {SEEDVR2_PATH}")
        time.sleep(10)
    else:
        raise RuntimeError("Could not clone SeedVR2")
else:
    print("✅ SeedVR2 already cloned")

print("📦 Installing SeedVR2 requirements...")
os.system(f"pip install -q -r {SEEDVR2_PATH}/requirements.txt")

SEEDVR2_MODEL_DIR = "/content/ComfyUI/models/SEEDVR2"
os.makedirs(SEEDVR2_MODEL_DIR, exist_ok=True)

DIT_MODEL = "seedvr2_ema_3b_fp8_e4m3fn.safetensors"
VAE_MODEL = "ema_vae_fp16.safetensors"
dit_path = f"{SEEDVR2_MODEL_DIR}/{DIT_MODEL}"
vae_path = f"{SEEDVR2_MODEL_DIR}/{VAE_MODEL}"

if not os.path.exists(dit_path):
    print(f"⏳ Downloading DiT model...")
    os.system(f'wget -c -q --show-progress -O "{dit_path}" "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/{DIT_MODEL}"')
else:
    print(f"✅ DiT ready")

if not os.path.exists(vae_path):
    print(f"⏳ Downloading VAE model...")
    os.system(f'wget -c -q --show-progress -O "{vae_path}" "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/{VAE_MODEL}"')
else:
    print(f"✅ VAE ready")

# ── Thread-based async runner ──
def _run_async(coro):
    result = [None]; exception = [None]
    def target():
        try:
            loop = asyncio.new_event_loop()
            try: result[0] = loop.run_until_complete(coro)
            finally: loop.close()
        except BaseException as e: exception[0] = e
    t = threading.Thread(target=target); t.start(); t.join()
    if exception[0] is not None: raise exception[0]
    return result[0]

def _run_maybe_async(fn, *args, **kwargs):
    if inspect.iscoroutinefunction(fn): return _run_async(fn(*args, **kwargs))
    result = fn(*args, **kwargs)
    if inspect.iscoroutine(result): return _run_async(result)
    return result

# ── Register SeedVR2 nodes ──
print("🔄 Registering SeedVR2 nodes...")
import nodes
if "SeedVR2VideoUpscaler" not in nodes.NODE_CLASS_MAPPINGS:
    if hasattr(nodes, 'init_extra_nodes'): _run_maybe_async(nodes.init_extra_nodes)
SeedVR2VideoUpscaler = nodes.NODE_CLASS_MAPPINGS.get("SeedVR2VideoUpscaler")
if not SeedVR2VideoUpscaler:
    raise RuntimeError("Failed to register SeedVR2")
print("✅ SeedVR2 nodes registered")

VAEEncode = NODE_CLASS_MAPPINGS.get("VAEEncode")
if VAEEncode:
    VAEEncode = VAEEncode()
    print("✅ VAEEncode loaded")

# ── PATCH: fix ComfyUI model GC crash (idempotent) ──
try:
    import comfy.model_management as _mm
    if hasattr(_mm, 'LoadedModel') and not getattr(_mm, '_is_dead_patched', False):
        _orig_is_dead = _mm.LoadedModel.is_dead
        def _safe_is_dead(self):
            try:
                if self.real_model is None or not callable(self.real_model): return True
                return _orig_is_dead(self)
            except (TypeError, AttributeError): return True
        _mm.LoadedModel.is_dead = _safe_is_dead
        _mm._is_dead_patched = True
        print("✅ Patched LoadedModel.is_dead")
    if hasattr(_mm, 'cleanup_models_gc') and not getattr(_mm, '_gc_patched', False):
        _orig_cleanup = _mm.cleanup_models_gc
        def _safe_cleanup():
            try: return _orig_cleanup()
            except TypeError:
                if hasattr(_mm, 'current_loaded_models'):
                    _mm.current_loaded_models = [m for m in _mm.current_loaded_models
                        if m is not None and hasattr(m, 'real_model')
                        and m.real_model is not None and callable(m.real_model)]
        _mm.cleanup_models_gc = _safe_cleanup
        _mm._gc_patched = True
        print("✅ Patched cleanup_models_gc")
except Exception as e:
    print(f"⚠️ GC patch failed: {e}")

# ── Stub execution context ──
_fake_ctx = MagicMock()
_fake_ctx.node_id = "seedvr2_colab_notebook"
_fake_ctx.prompt_id = "seedvr2_colab_prompt"
def _fake_get_executing_context(): return _fake_ctx

_patched_modules = 0
for _mod_name in list(sys.modules.keys()):
    _mod = sys.modules.get(_mod_name)
    if _mod is None: continue
    try:
        if hasattr(_mod, 'get_executing_context'):
            setattr(_mod, 'get_executing_context', _fake_get_executing_context)
            _patched_modules += 1
    except Exception: continue
print(f"✅ Patched get_executing_context in {_patched_modules} modules")

try:
    from server import PromptServer
    if not hasattr(PromptServer, 'instance') or PromptServer.instance is None:
        _fake_server = MagicMock()
        _fake_server.client_id = None
        _fake_server.last_node_id = "seedvr2_colab"
        _fake_server.send_sync = lambda *a, **k: None
        _fake_server.send_progress_text = lambda *a, **k: None
        PromptServer.instance = _fake_server
        print("✅ Stubbed PromptServer.instance")
except Exception as e:
    print(f"⚠️ PromptServer stub skipped: {e}")

# ── Build config dicts ──
print("⏳ Building SeedVR2 config...")
seedvr2_dit_config = {
    "model": DIT_MODEL, "device": "cuda:0", "offload_device": "cpu",
    "cache_model": True, "blocks_to_swap": 20, "swap_io_components": False,
    "attention_mode": "sdpa", "torch_compile_args": None, "node_id": "seedvr2_colab_dit",
}
seedvr2_vae_config = {
    "model": VAE_MODEL, "device": "cuda:0", "offload_device": "cpu",
    "cache_model": True, "encode_tiled": True, "encode_tile_size": 512,
    "encode_tile_overlap": 256, "decode_tiled": True, "decode_tile_size": 512,
    "decode_tile_overlap": 256, "tile_debug": "false", "torch_compile_args": None,
    "node_id": "seedvr2_colab_vae",
}
print("✅ SeedVR2 configured")

# ── Helpers ──
def _call_execute(node_cls, **kwargs):
    try: fn = node_cls.execute; result = fn(**kwargs)
    except TypeError: inst = node_cls(); result = inst.execute(**kwargs)
    if inspect.iscoroutine(result): result = _run_async(result)
    return result

def _unwrap_tensor(obj):
    if isinstance(obj, (torch.Tensor, np.ndarray)): return obj
    if hasattr(obj, 'values'):
        val = obj.values
        if isinstance(val, (list, tuple)) and len(val) > 0: return _unwrap_tensor(val[0])
        return _unwrap_tensor(val)
    if hasattr(obj, '__getitem__'):
        try: return _unwrap_tensor(obj[0])
        except: pass
    if hasattr(obj, 'data'): return _unwrap_tensor(obj.data)
    if hasattr(obj, 'args'):
        args = obj.args
        if isinstance(args, (list, tuple)) and len(args) > 0: return _unwrap_tensor(args[0])
        return _unwrap_tensor(args)
    return obj

def _call_node_2b(node_instance, **kwargs):
    fn_name = node_instance.FUNCTION if hasattr(node_instance, 'FUNCTION') else type(node_instance).FUNCTION
    return getattr(node_instance, fn_name)(**kwargs)

# ── Before / After storage ──
_upscale_before = None
_upscale_after  = None
def _get_comparison_images():
    return _upscale_before, _upscale_after

# ── Refine function (with optional dedicated LoRA) ──
@torch.inference_mode()
def _refine_upscaled(pil_image, prompt, negative_prompt, refine_denoise, refine_steps,
                     use_enhanced_pass, refine_lora_name=None, refine_lora_strength=0.8):
    _vae = vae; _unet = active_unet; _clip = active_clip

    if refine_lora_name and refine_lora_name not in ("(none)", "(no LoRAs found)", ""):
        try:
            filename = _symlink_lora(refine_lora_name.strip())
            _unet, _, info = _apply_lora_standard(_unet, _clip, filename, float(refine_lora_strength), 0.0, tag="refine")
            print(f"{'✅' if info['ok'] else '❌'} Refine LoRA: {filename} (str: {refine_lora_strength}) — {info['matched']} patches" + (f" | {info['error']}" if info['error'] else ""))
        except Exception as e:
            print(f"⚠️ Refine LoRA failed: {e}")

    arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(arr).unsqueeze(0)
    if not VAEEncode: raise RuntimeError("VAEEncode not available")
    latent = VAEEncode.encode(_vae, img_tensor)[0]
    positive = CLIPTextEncode.encode(_clip, prompt)[0]
    negative = CLIPTextEncode.encode(_clip, negative_prompt)[0]
    seed = random.randint(0, 2**63 - 1)

    if use_enhanced_pass and all([_ModelSamplingZImage, _LGNoiseInjectionLatent, _GenerateNoise]):
        model_zimage = _call_node_2b(_ModelSamplingZImage, model=_unet, shift=3, multiplier=1)[0]
        w, h = pil_image.size
        noise_latent = _call_node_2b(_GenerateNoise, width=w, height=h, batch_size=1,
            seed=seed, multiplier=1.0, constant_batch_noise=False, normalize=False,
            latent_channels="16", shape="BCHW", model=model_zimage, sigmas=None)[0]
        model_injected = _call_node_2b(_LGNoiseInjectionLatent, model=model_zimage,
            reference_latent=noise_latent, strength=0.25, start_percent=0.0, end_percent=0.15)[0]
        pass1 = KSampler.sample(model_injected, seed, max(refine_steps, 4), 1.0,
            "euler", "simple", positive, negative, latent, denoise=float(refine_denoise))[0]
        try:
            refined = KSampler.sample(model_injected, seed+1, 3, 1.0, "res_multistep", "simple",
                positive, negative, pass1, denoise=float(refine_denoise)*0.5)[0]
        except:
            refined = KSampler.sample(model_injected, seed+1, 3, 1.0, "dpmpp_2m_sde", "simple",
                positive, negative, pass1, denoise=float(refine_denoise)*0.5)[0]
    else:
        refined = KSampler.sample(_unet, seed, int(refine_steps), 1.0, "euler", "simple",
            positive, negative, latent, denoise=float(refine_denoise))[0]

    decoded = VAEDecode.decode(_vae, refined)[0].detach()
    return _tensor_to_pil(decoded, 0)

# ── Detail Daemon Sampler (ported from ComfyUI-Detail-Daemon by Jonseed/muerrilla/blepping) ──
# Uses the exact same node chain as the ComfyUI workflow:
#   KSamplerSelect → DetailDaemonSamplerNode → SamplerCustomAdvanced
#   BasicScheduler → sigmas
#   BasicGuider(model, conditioning)
# This is NOT sample_custom — it's the proper guider→sampler→model chain.

from comfy.samplers import KSAMPLER
import comfy.samplers

def _make_detail_daemon_schedule(steps, start, end, bias, amount, exponent,
                                  start_offset, end_offset, fade, smooth):
    start = min(start, end)
    mid = start + bias * (end - start)
    multipliers = np.zeros(steps)
    start_idx, mid_idx, end_idx = [int(round(x * (steps - 1))) for x in [start, mid, end]]
    start_values = np.linspace(0, 1, mid_idx - start_idx + 1)
    if smooth:
        start_values = 0.5 * (1 - np.cos(start_values * np.pi))
    start_values = start_values ** exponent
    if start_values.any():
        start_values *= amount - start_offset
        start_values += start_offset
    end_values = np.linspace(1, 0, end_idx - mid_idx + 1)
    if smooth:
        end_values = 0.5 * (1 - np.cos(end_values * np.pi))
    end_values = end_values ** exponent
    if end_values.any():
        end_values *= amount - end_offset
        end_values += end_offset
    multipliers[start_idx:mid_idx + 1] = start_values
    multipliers[mid_idx:end_idx + 1] = end_values
    multipliers[:start_idx] = start_offset
    multipliers[end_idx + 1:] = end_offset
    multipliers *= 1 - fade
    return multipliers

def _get_dd_schedule(sigma, sigmas, dd_schedule):
    sched_len = len(dd_schedule)
    if sched_len < 2 or len(sigmas) < 2 or sigma <= 0 or not (sigmas[-1] <= sigma <= sigmas[0]):
        return 0.0
    deltas = (sigmas[:-1] - sigma).abs()
    idx = int(deltas.argmin())
    if (idx == 0 and sigma >= sigmas[0]) or (idx == sched_len - 1 and sigma <= sigmas[-2]) or deltas[idx] == 0:
        return dd_schedule[idx].item()
    idxlow, idxhigh = (idx, idx - 1) if sigma > sigmas[idx] else (idx + 1, idx)
    nlow, nhigh = sigmas[idxlow], sigmas[idxhigh]
    if nhigh - nlow == 0:
        return dd_schedule[idxlow]
    ratio = ((sigma - nlow) / (nhigh - nlow)).clamp(0, 1)
    return torch.lerp(dd_schedule[idxlow], dd_schedule[idxhigh], ratio).item()

def _detail_daemon_sampler_fn(model, x, sigmas, *, dds_wrapped_sampler,
                               dds_make_schedule, dds_cfg_scale_override, **kwargs):
    """Exact port of detail_daemon_sampler from the DD node."""
    if dds_cfg_scale_override > 0:
        cfg_scale = dds_cfg_scale_override
    else:
        maybe_cfg_scale = getattr(model, "cfg", None)
        if maybe_cfg_scale is None:
            im = getattr(model, "inner_model", None)
            if im is not None:
                maybe_cfg_scale = getattr(im, "cfg", None)
        cfg_scale = float(maybe_cfg_scale) if isinstance(maybe_cfg_scale, (int, float)) else 1.0

    dd_schedule = torch.tensor(dds_make_schedule(len(sigmas) - 1), dtype=torch.float32, device="cpu")
    sigmas_cpu = sigmas.detach().clone().cpu()
    sigma_max, sigma_min = float(sigmas_cpu[0]), float(sigmas_cpu[-1]) + 1e-05

    def model_wrapper(x, sigma, **extra_args):
        sigma_float = float(sigma.max().detach().cpu())
        if not (sigma_min <= sigma_float <= sigma_max):
            return model(x, sigma, **extra_args)
        dd_adjustment = _get_dd_schedule(sigma_float, sigmas_cpu, dd_schedule) * 0.1
        adjusted_sigma = sigma * max(1e-06, 1.0 - dd_adjustment * cfg_scale)
        return model(x, adjusted_sigma, **extra_args)

    for k in ("inner_model", "sigmas"):
        if hasattr(model, k):
            setattr(model_wrapper, k, getattr(model, k))

    return dds_wrapped_sampler.sampler_function(model_wrapper, x, sigmas,
                                                 **kwargs, **dds_wrapped_sampler.extra_options)

@torch.inference_mode()
def _detail_daemon_pass(pil_image, prompt, negative_prompt,
                        dd_amount=0.1, dd_start=0.2, dd_end=0.8,
                        dd_bias=0.5, dd_exponent=1.0,
                        dd_start_offset=0.0, dd_end_offset=0.0,
                        dd_fade=0.0, dd_smooth=True,
                        dd_cfg_override=1.0,
                        dd_denoise=0.25, dd_steps=6):
    """
    Post-refine detail enhancement using the exact Detail Daemon Sampler flow:
      BasicScheduler → sigmas
      KSamplerSelect → base sampler
      DetailDaemonSampler(base_sampler, params) → wrapped sampler
      BasicGuider(model, conditioning) → guider
      SamplerCustomAdvanced(noise, guider, dd_sampler, sigmas, latent) → output
    """
    _vae = vae; _unet = active_unet; _clip = active_clip

    # Encode image to latent
    arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(arr).unsqueeze(0)
    latent = VAEEncode.encode(_vae, img_tensor)[0]
    positive = CLIPTextEncode.encode(_clip, prompt)[0]
    seed = random.randint(0, 2**63 - 1)

    steps = int(dd_steps)
    denoise = float(dd_denoise)
    cfg_override = float(dd_cfg_override)

    print(f"\U0001f50d Detail Daemon: amount={dd_amount}, start={dd_start}, end={dd_end}, "
          f"bias={dd_bias}, exponent={dd_exponent}, steps={steps}, denoise={denoise}")

    # ── Step 1: BasicScheduler → sigmas ──
    # Matches: BasicScheduler(model, "simple", steps=10, denoise=0.6)
    total_steps = steps
    if denoise < 1.0 and denoise > 0.0:
        total_steps = int(steps / denoise)
    sigmas = comfy.samplers.calculate_sigmas(
        _unet.get_model_object("model_sampling"), "simple", total_steps).cpu()
    sigmas = sigmas[-(steps + 1):]

    # ── Step 2: KSamplerSelect("euler") → base sampler ──
    base_sampler = comfy.samplers.sampler_object("euler")

    # ── Step 3: DetailDaemonSamplerNode(base_sampler, params) → DD sampler ──
    def dds_make_schedule(num_steps):
        return _make_detail_daemon_schedule(
            num_steps, float(dd_start), float(dd_end), float(dd_bias),
            float(dd_amount), float(dd_exponent),
            float(dd_start_offset), float(dd_end_offset),
            float(dd_fade), bool(dd_smooth))

    dd_sampler = KSAMPLER(
        _detail_daemon_sampler_fn,
        extra_options={
            "dds_wrapped_sampler": base_sampler,
            "dds_make_schedule": dds_make_schedule,
            "dds_cfg_scale_override": cfg_override,
        })

    # ── Step 4: BasicGuider(model, conditioning) → guider ──
    # Guider_Basic extends CFGGuider — wraps model for guided sampling
    from comfy_extras.nodes_custom_sampler import Guider_Basic
    guider = Guider_Basic(_unet)
    guider.set_conds(positive)

    # ── Step 5: SamplerCustomAdvanced(noise, guider, dd_sampler, sigmas, latent) ──
    latent_image = latent["samples"]
    latent_image = comfy.sample.fix_empty_latent_channels(guider.model_patcher, latent_image)

    # Generate noise (matches RandomNoise node)
    noise = comfy.sample.prepare_noise(latent_image, seed)

    # Run sampling through guider → DD sampler → model
    samples = guider.sample(noise, latent_image, dd_sampler, sigmas,
                            denoise_mask=None, callback=None, disable_pbar=True, seed=seed)
    samples = samples.to(comfy.model_management.intermediate_device())

    decoded = VAEDecode.decode(_vae, {"samples": samples})[0].detach()
    return _tensor_to_pil(decoded, 0)

def _run_seedvr2(pil_image, resolution, color_correction, input_noise, latent_noise):
    arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).unsqueeze(0)
    torch.cuda.empty_cache()
    result = _call_execute(SeedVR2VideoUpscaler, image=tensor, dit=seedvr2_dit_config,
        vae=seedvr2_vae_config, seed=random.randint(0, 2**31-1), resolution=int(resolution),
        max_resolution=int(resolution), batch_size=1, uniform_batch_size=False,
        temporal_overlap=0, prepend_frames=0, color_correction=color_correction,
        input_noise_scale=float(input_noise), latent_noise_scale=float(latent_noise),
        offload_device="cpu", enable_debug=False)
    output_tensor = _unwrap_tensor(result)
    output_np = output_tensor.cpu().numpy() if isinstance(output_tensor, torch.Tensor) else output_tensor
    while output_np.ndim > 3: output_np = output_np[0]
    output_np = (output_np * 255.0).clip(0, 255).astype(np.uint8)
    return Image.fromarray(output_np)

# ── FameGrid Auto Color (applied after refine) ──
@torch.inference_mode()
def _apply_auto_color(pil_image, strength=1.10, protect_skin=True):
    if _FameGridAutoColor is None:
        print("⚠️ FameGrid Auto Color not loaded — skipping")
        return pil_image
    arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).unsqueeze(0)
    try:
        fn_name = _FameGridAutoColor.FUNCTION if hasattr(_FameGridAutoColor, 'FUNCTION') else type(_FameGridAutoColor).FUNCTION
        result = getattr(_FameGridAutoColor, fn_name)(
            image=tensor,
            white_balance_power=8,
            auto_color_strength=float(strength),
            correct_contrast=True,
            contrast_clip_percent=7.3,
            normalize_saturation=True,
            saturation_strength=0.15,
            protect_skin=protect_skin,
            brightness=0.10,
            shadows=-0.15,
            highlights=-0.05,
            saturation=0.0,
            vibrance=-0.35,
        )
        output = result[0] if isinstance(result, tuple) else result
        if isinstance(output, dict):
            output = list(output.values())[0]
        if isinstance(output, torch.Tensor):
            out_np = output.cpu().numpy()
            while out_np.ndim > 3: out_np = out_np[0]
            out_np = (out_np * 255.0).clip(0, 255).astype(np.uint8)
            return Image.fromarray(out_np)
        return pil_image
    except Exception as e:
        print(f"⚠️ Auto Color failed: {e}")
        return pil_image

# ── Main upscale pipeline ──
import random

@torch.inference_mode()
def upscale_image(pil_image, resolution, color_correction, enable_debug,
                  pre_downscale, input_noise, latent_noise,
                  refine_enabled, refine_prompt, refine_negative,
                  refine_denoise, refine_steps, refine_enhanced,
                  refine_lora_name, refine_lora_strength,
                  auto_color_enabled, auto_color_strength, auto_color_protect_skin,
                  final_upscale_enabled, final_resolution,
                  dd_enabled=False, dd_amount=0.1, dd_start=0.2, dd_end=0.8,
                  dd_bias=0.5, dd_exponent=1.0, dd_start_offset=0.0, dd_end_offset=0.0,
                  dd_fade=0.0, dd_smooth=True, dd_cfg_override=1.0,
                  dd_denoise=0.25, dd_steps=6):
    global _upscale_before, _upscale_after

    if pil_image is None:
        yield None, "❌ No image provided."
        return

    original_input = pil_image.copy()
    yield None, "⏳ Preparing..."

    pre_downscale = float(pre_downscale)
    if pre_downscale < 1.0:
        new_w = max(64, int(pil_image.width * pre_downscale))
        new_h = max(64, int(pil_image.height * pre_downscale))
        pil_image = pil_image.resize((new_w, new_h), Image.LANCZOS)

    total = 1 + int(refine_enabled) + int(dd_enabled) + int(auto_color_enabled) + int(final_upscale_enabled)
    step = 1

    target_res = int(resolution)
    TWO_PASS_THRESHOLD = 2560  # above this, split into 2 passes

    if target_res > TWO_PASS_THRESHOLD:
        # Two-pass: first to ~2K, then from 2K to target
        mid_res = min(2048, TWO_PASS_THRESHOLD)
        yield None, f"⏳ [{step}/{total}] SeedVR2 pass 1/2 → {mid_res}px..."
        try:
            upscaled_pil = _run_seedvr2(pil_image, mid_res, color_correction, input_noise, latent_noise)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache(); yield None, f"❌ OOM at {mid_res}px (pass 1)!"; return
            raise
        torch.cuda.empty_cache()
        yield upscaled_pil, f"⏳ [{step}/{total}] SeedVR2 pass 2/2 → {target_res}px..."
        try:
            upscaled_pil = _run_seedvr2(upscaled_pil, target_res, color_correction, 0.02, 0.0)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache()
                yield upscaled_pil, f"⚠️ OOM at {target_res}px (pass 2) — returning {mid_res}px result";
                # Continue with mid_res result instead of failing
            else: raise
        print(f"✅ SeedVR2 two-pass: {pil_image.size} → {mid_res}px → {upscaled_pil.size}")
    else:
        yield None, f"⏳ [{step}/{total}] SeedVR2 upscale → {target_res}px..."
        try:
            upscaled_pil = _run_seedvr2(pil_image, target_res, color_correction, input_noise, latent_noise)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache(); yield None, f"❌ OOM at {target_res}px!"; return
            raise

    parts = []

    if refine_enabled:
        step += 1
        lora_label = ""
        if refine_lora_name and refine_lora_name not in ("(none)", "(no LoRAs found)", ""):
            lora_label = f" + {refine_lora_name.split('.')[0]}"
        yield upscaled_pil, f"⏳ [{step}/{total}] Z-Image refine{lora_label}..."
        torch.cuda.empty_cache()
        prompt = refine_prompt.strip() or "high quality, detailed, sharp"
        neg = refine_negative.strip() or "blurry ugly bad"
        try:
            upscaled_pil = _refine_upscaled(upscaled_pil, prompt, neg,
                refine_denoise, refine_steps, refine_enhanced,
                refine_lora_name=refine_lora_name, refine_lora_strength=refine_lora_strength)
            parts.append("Refined" + lora_label)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); parts.append("Refine OOM")
        except Exception as e:
            parts.append("Refine err"); print(f"⚠️ Refine: {e}")

    if dd_enabled:
        step += 1
        yield upscaled_pil, f"⏳ [{step}/{total}] Detail Daemon (amount:{dd_amount})..."
        torch.cuda.empty_cache()
        prompt = refine_prompt.strip() if refine_enabled else "high quality, detailed, sharp"
        neg = refine_negative.strip() if refine_enabled else "blurry ugly bad"
        try:
            upscaled_pil = _detail_daemon_pass(upscaled_pil, prompt, neg,
                dd_amount=dd_amount, dd_start=dd_start, dd_end=dd_end,
                dd_bias=dd_bias, dd_exponent=dd_exponent,
                dd_start_offset=dd_start_offset, dd_end_offset=dd_end_offset,
                dd_fade=dd_fade, dd_smooth=dd_smooth,
                dd_cfg_override=dd_cfg_override,
                dd_denoise=dd_denoise, dd_steps=dd_steps)
            parts.append(f"DD×{dd_amount}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); parts.append("DD OOM")
        except Exception as e:
            parts.append("DD err"); print(f"⚠️ Detail Daemon: {e}")

    if auto_color_enabled and _FameGridAutoColor:
        step += 1
        yield upscaled_pil, f"⏳ [{step}/{total}] Auto color correction..."
        try:
            upscaled_pil = _apply_auto_color(upscaled_pil, auto_color_strength, auto_color_protect_skin)
            parts.append("Auto Color")
        except Exception as e:
            parts.append("Color err"); print(f"⚠️ Auto Color: {e}")

    if final_upscale_enabled:
        step += 1
        target = int(final_resolution)
        yield upscaled_pil, f"⏳ [{step}/{total}] Final upscale → {target}px (Lanczos)..."
        torch.cuda.empty_cache()
        # Simple Lanczos upscale — does NOT run SeedVR2 again (which degrades at 4K)
        cur_w, cur_h = upscaled_pil.size
        cur_max = max(cur_w, cur_h)
        if target > cur_max:
            ratio = target / cur_max
            new_w = int(round(cur_w * ratio / 8)) * 8
            new_h = int(round(cur_h * ratio / 8)) * 8
            upscaled_pil = upscaled_pil.resize((new_w, new_h), Image.LANCZOS)
            parts.append(f"Final {new_w}×{new_h}")
        else:
            parts.append("Final (already at target)")

    _upscale_before = original_input.resize(upscaled_pil.size, Image.LANCZOS)
    _upscale_after = upscaled_pil

    ts = int(time.time())
    upscaled_pil.save(f"{OUTPUT_DIR}/upscaled_{ts}.png")

    suffix = " + ".join(parts) if parts else ""
    status = f"✅ {upscaled_pil.size[0]}×{upscaled_pil.size[1]}"
    if suffix: status += f" ({suffix})"
    yield upscaled_pil, status + " → saved"

print("🎉 SeedVR2 upscaler ready!")

In [ ]:
# @title Cell 2C — Custom Nodes

# ════════════════════════════════════════════════════════════════════
# CELL 2C — LOAD CUSTOM NODES
# LG_SamplingUtils, KJNodes, CapitanZiT, FameGrid Auto Color,
# Z-Image Power Nodes, embedded SceneComposer
# Run after Cell 2B, before Cell 3A/3B
# ════════════════════════════════════════════════════════════════════

import os, sys, importlib, importlib.util, math
import torch
import torch.nn.functional as F
import nodes

LG_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-LG_SamplingUtils"
KJ_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-KJNodes"
ZIT_SCHED_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-CapitanZiT-Scheduler"
FG_PATH = "/content/ComfyUI/custom_nodes/famegrid-auto-color"
ZPOW_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-ZImagePowerNodes"

if not os.path.exists(f"{LG_PATH}/__init__.py"):
    raise RuntimeError("LG_SamplingUtils not found. Re-run Cell 1.")
if not os.path.exists(f"{KJ_PATH}/__init__.py"):
    raise RuntimeError("KJNodes not found. Re-run Cell 1.")

def _load_custom_node_package(pkg_path, pkg_name):
    parent = os.path.dirname(pkg_path)
    if parent not in sys.path: sys.path.insert(0, parent)
    folder_name = os.path.basename(pkg_path)
    init_file = os.path.join(pkg_path, "__init__.py")
    for key in list(sys.modules.keys()):
        if folder_name.replace("-", "_").lower() in key.lower() or pkg_name.lower() in key.lower():
            del sys.modules[key]
    try:
        spec = importlib.util.spec_from_file_location(
            folder_name, init_file, submodule_search_locations=[pkg_path])
        mod = importlib.util.module_from_spec(spec)
        sys.modules[folder_name] = mod
        spec.loader.exec_module(mod)
        count = 0
        if hasattr(mod, 'NODE_CLASS_MAPPINGS') and mod.NODE_CLASS_MAPPINGS:
            nodes.NODE_CLASS_MAPPINGS.update(mod.NODE_CLASS_MAPPINGS)
            count = len(mod.NODE_CLASS_MAPPINGS)
        if hasattr(mod, 'NODE_DISPLAY_NAME_MAPPINGS') and hasattr(nodes, 'NODE_DISPLAY_NAME_MAPPINGS'):
            nodes.NODE_DISPLAY_NAME_MAPPINGS.update(mod.NODE_DISPLAY_NAME_MAPPINGS)
        return count
    except Exception as e:
        print(f"⚠️  Error loading {folder_name}: {e}")
        import traceback; traceback.print_exc()
        return 0

def _load_custom_node_file(filepath, module_name):
    try:
        spec = importlib.util.spec_from_file_location(module_name, filepath)
        mod = importlib.util.module_from_spec(spec)
        sys.modules[module_name] = mod
        spec.loader.exec_module(mod)
        count = 0
        if hasattr(mod, 'NODE_CLASS_MAPPINGS') and mod.NODE_CLASS_MAPPINGS:
            nodes.NODE_CLASS_MAPPINGS.update(mod.NODE_CLASS_MAPPINGS)
            count = len(mod.NODE_CLASS_MAPPINGS)
        if hasattr(mod, 'NODE_DISPLAY_NAME_MAPPINGS') and hasattr(nodes, 'NODE_DISPLAY_NAME_MAPPINGS'):
            nodes.NODE_DISPLAY_NAME_MAPPINGS.update(mod.NODE_DISPLAY_NAME_MAPPINGS)
        return count
    except Exception as e:
        print(f"⚠️  Error loading {module_name}: {e}")
        return 0

# ── Register LG_SamplingUtils ──
print("🔄 Registering LG_SamplingUtils...")
lg_count = _load_custom_node_package(LG_PATH, "lg_samplingutils")
print(f"   → {lg_count} nodes" if lg_count else "   → ⚠️ Failed")

# ── Register KJNodes ──
print("🔄 Registering KJNodes...")
kj_count = _load_custom_node_package(KJ_PATH, "kjnodes")
print(f"   → {kj_count} nodes" if kj_count else "   → ⚠️ Failed")

# ── Register CapitanZiT Scheduler ──
if os.path.exists(ZIT_SCHED_PATH):
    print("🔄 Registering CapitanZiT Scheduler...")
    for fname, mname in [("capitan_zit_scheduler.py", "capitan_zit_scheduler"),
                          ("smooth_cosine_scheduler.py", "smooth_cosine_scheduler"),
                          ("minimal_change_sampler.py", "minimal_change_sampler")]:
        fpath = os.path.join(ZIT_SCHED_PATH, fname)
        if os.path.exists(fpath):
            c = _load_custom_node_file(fpath, mname)
            print(f"   → {mname}: {c} nodes" if c else f"   → ⚠️ {mname} failed")
else:
    print("⚠️ CapitanZiT not found")

# ── Register FameGrid Auto Color ──
_FameGridAutoColor = None

if os.path.exists(FG_PATH):
    print("🔄 Registering FameGrid Auto Color...")
    fg_count = _load_custom_node_package(FG_PATH, "famegrid_auto_color")
    if fg_count:
        for name, cls in nodes.NODE_CLASS_MAPPINGS.items():
            if "famegrid" in name.lower() or ("auto" in name.lower() and "color" in name.lower()):
                _FameGridAutoColor = cls()
                print(f"✅ FameGrid Auto Color loaded: {name}")
                break
        if _FameGridAutoColor is None:
            try:
                spec = importlib.util.spec_from_file_location("fg_node", os.path.join(FG_PATH, "node.py"))
                fg_mod = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(fg_mod)
                if hasattr(fg_mod, 'NODE_CLASS_MAPPINGS') and fg_mod.NODE_CLASS_MAPPINGS:
                    _FameGridAutoColor = list(fg_mod.NODE_CLASS_MAPPINGS.values())[0]()
                    print("✅ FameGrid Auto Color loaded (fallback)")
            except Exception as e:
                print(f"   → ⚠️ Fallback failed: {e}")
    if _FameGridAutoColor is None:
        print(f"   → ⚠️ FameGrid registration failed")
else:
    print("⚠️ FameGrid Auto Color not found")

# ── Register Z-Image Power Nodes ──
_ZSamplerTurbo = None
_ZSamplerTurboSimple = None
_StylePromptEncoder = None
_StyleStringInjector = None
_MyTop10Styles = None
_VAEEncodeForSoftInpainting = None
_EmptyZImageLatentImage = None
_ZImageSaveImage = None
_ZPOW_STYLES = []

if os.path.exists(f"{ZPOW_PATH}/__init__.py"):
    print("🔄 Registering Z-Image Power Nodes...")
    zpow_count = _load_custom_node_package(ZPOW_PATH, "zimage_power_nodes")
    if zpow_count:
        # Grab all available node instances
        for node_name, cls in nodes.NODE_CLASS_MAPPINGS.items():
            nl = node_name.lower()
            if "zsampler" in nl and "simple" in nl:
                _ZSamplerTurboSimple = cls()
                print(f"   → ✅ {node_name}")
            elif "zsampler" in nl and "simple" not in nl:
                _ZSamplerTurbo = cls()
                print(f"   → ✅ {node_name}")
            elif "style" in nl and "prompt" in nl and "encoder" in nl:
                _StylePromptEncoder = cls()
                print(f"   → ✅ {node_name}")
            elif "style" in nl and "string" in nl and "inject" in nl:
                _StyleStringInjector = cls()
                print(f"   → ✅ {node_name}")
            elif "top" in nl and "10" in nl and "style" in nl:
                _MyTop10Styles = cls()
                print(f"   → ✅ {node_name}")
            elif "vae" in nl and "soft" in nl and "inpaint" in nl:
                _VAEEncodeForSoftInpainting = cls()
                print(f"   → ✅ {node_name}")
            elif "empty" in nl and "zimage" in nl and "latent" in nl:
                _EmptyZImageLatentImage = cls()
                print(f"   → ✅ {node_name}")
            elif "save" in nl and "image" in nl and ("zimage" in nl or "power" in nl):
                _ZImageSaveImage = cls()
                print(f"   → ✅ {node_name}")

        # Try to get styles list
        try:
            if _StylePromptEncoder and hasattr(_StylePromptEncoder, 'INPUT_TYPES'):
                input_spec = _StylePromptEncoder.INPUT_TYPES()
                if isinstance(input_spec, dict) and 'required' in input_spec:
                    for k, v in input_spec['required'].items():
                        if 'style' in k.lower() and isinstance(v, (list, tuple)):
                            if isinstance(v[0], (list, tuple)):
                                _ZPOW_STYLES = list(v[0])
                            else:
                                _ZPOW_STYLES = list(v)
                            break
        except Exception as e:
            print(f"   → ⚠️ Could not extract styles: {e}")

        # Also try from StyleStringInjector
        if not _ZPOW_STYLES and _StyleStringInjector:
            try:
                input_spec = _StyleStringInjector.INPUT_TYPES()
                if isinstance(input_spec, dict) and 'required' in input_spec:
                    for k, v in input_spec['required'].items():
                        if 'style' in k.lower() and isinstance(v, (list, tuple)):
                            if isinstance(v[0], (list, tuple)):
                                _ZPOW_STYLES = list(v[0])
                            else:
                                _ZPOW_STYLES = list(v)
                            break
            except:
                pass

        print(f"✅ Z-Image Power Nodes: {zpow_count} total" +
              (f" ({len(_ZPOW_STYLES)} styles)" if _ZPOW_STYLES else ""))

        # List all registered power nodes
        zpow_all = [k for k in nodes.NODE_CLASS_MAPPINGS.keys()
                     if any(x in k.lower() for x in ["zimage", "z_image", "zsampler", "power"])]
        if zpow_all:
            print(f"   Nodes: {', '.join(zpow_all)}")
    else:
        print("   → ⚠️ Registration failed (0 nodes)")
else:
    print("⚠️ Z-Image Power Nodes not found — run Cell 1")

# ── Register Inpaint CropAndStitch ──
CS_PATH = f"{CUSTOM_NODES}/ComfyUI-Inpaint-CropAndStitch"
if os.path.exists(f"{CS_PATH}/__init__.py"):
    print("🔄 Registering Inpaint CropAndStitch...")
    cs_count = _load_custom_node_package(CS_PATH, "inpaint_cropandstitch")
    if cs_count:
        print(f"✅ Inpaint CropAndStitch: {cs_count} nodes registered")
    else:
        print("   → ⚠️ CropAndStitch registration failed")
else:
    print("⚠️ Inpaint CropAndStitch not found — run Cell 1")

# ── Register ControlNet Aux (DWPose) ──
CNAUX_PATH = f"{CUSTOM_NODES}/comfyui_controlnet_aux"
_DWPose = None
if os.path.exists(f"{CNAUX_PATH}/__init__.py"):
    print("🔄 Registering ControlNet Aux...")
    cnaux_count = _load_custom_node_package(CNAUX_PATH, "comfyui_controlnet_aux")
    _dw_cls = nodes.NODE_CLASS_MAPPINGS.get("DWPreprocessor")
    if _dw_cls:
        _DWPose = _dw_cls()
        print(f"✅ ControlNet Aux: {cnaux_count} nodes | DWPose ready")
    else:
        print("   → ⚠️ DWPreprocessor not found in registered nodes")
else:
    print("⚠️ ControlNet Aux not found — run Cell 1")

# ── Register ComfyUI-GGUF (for Qwen Edit) ──
GGUF_PATH = f"{CUSTOM_NODES}/ComfyUI-GGUF"
if os.path.exists(f"{GGUF_PATH}/__init__.py"):
    print("🔄 Registering ComfyUI-GGUF...")
    gguf_count = _load_custom_node_package(GGUF_PATH, "ComfyUI_GGUF")
    if nodes.NODE_CLASS_MAPPINGS.get("UnetLoaderGGUF"):
        print(f"✅ ComfyUI-GGUF: {gguf_count} nodes | UnetLoaderGGUF ready")
    else:
        print("   → ⚠️ UnetLoaderGGUF not found after registration")
else:
    print("⚠️ ComfyUI-GGUF not found — set DOWNLOAD_QWEN_EDIT=True in Cell 1")

# ── Z-Image ControlNet Union (model patch) ──
_ModelPatchLoader = None
_ZImageFunControlnet = None
_CN_UNION_NAME = "Z-Image-Turbo-Fun-Controlnet-Union-2.1-2602-8steps.safetensors"
_cn_union_patch = None
try:
    _mpl_cls = nodes.NODE_CLASS_MAPPINGS.get("ModelPatchLoader")
    _zfc_cls = nodes.NODE_CLASS_MAPPINGS.get("ZImageFunControlnet")
    if _mpl_cls and _zfc_cls:
        _ModelPatchLoader = _mpl_cls()
        _ZImageFunControlnet = _zfc_cls()
        patch_path = f"/content/ComfyUI/models/model_patches/{_CN_UNION_NAME}"
        if os.path.exists(patch_path):
            _cn_union_patch = _ModelPatchLoader.load_model_patch(_CN_UNION_NAME)[0]
            print("✅ Z-Image ControlNet Union loaded (pose/depth/canny)")
        else:
            print("⚠️ ControlNet Union file not found — run Cell 1")
    else:
        print("⚠️ ModelPatchLoader / ZImageFunControlnet not in ComfyUI — update ComfyUI")
except Exception as e:
    print(f"⚠️ ControlNet Union load failed: {e}")

# ── Grab texture enhancement node instances ──
_ModelSamplingZImage_cls    = nodes.NODE_CLASS_MAPPINGS.get("ModelSamplingZImage")
_LGNoiseInjectionLatent_cls = nodes.NODE_CLASS_MAPPINGS.get("LGNoiseInjectionLatent")
_GenerateNoise_cls          = nodes.NODE_CLASS_MAPPINGS.get("GenerateNoise")

if all([_ModelSamplingZImage_cls, _LGNoiseInjectionLatent_cls, _GenerateNoise_cls]):
    _ModelSamplingZImage     = _ModelSamplingZImage_cls()
    _LGNoiseInjectionLatent  = _LGNoiseInjectionLatent_cls()
    _GenerateNoise           = _GenerateNoise_cls()
    print("✅ Enhanced texture mode AVAILABLE")
else:
    missing = []
    if not _ModelSamplingZImage_cls: missing.append("ModelSamplingZImage")
    if not _LGNoiseInjectionLatent_cls: missing.append("LGNoiseInjectionLatent")
    if not _GenerateNoise_cls: missing.append("GenerateNoise")
    print(f"⚠️  Missing: {', '.join(missing)} — Enhanced texture NOT available")

# ── CapitanZiT summary ──
zit_nodes = [k for k in nodes.NODE_CLASS_MAPPINGS.keys()
             if any(x in k.lower() for x in ["capitan", "zit", "smoothcosine", "minimalchange"])]
if zit_nodes:
    print(f"✅ CapitanZiT nodes: {', '.join(zit_nodes)}")

# ════════════════════════════════════════════════════════════════════
# EMBEDDED SCENECOMPOSER — LatentCompositionGuide
# Shapes initial noise to guide subject placement & layout
# NOTE: Latent noise shaping doesn't work with flow-matching models
# Kept for reference but composition uses prompt-based approach instead
# ════════════════════════════════════════════════════════════════════

print("🔄 Building SceneComposer (embedded)...")

def _sc_grid(h, w, device):
    ys = torch.linspace(0, 1, h, device=device).view(h, 1).expand(h, w)
    xs = torch.linspace(0, 1, w, device=device).view(1, w).expand(h, w)
    return ys, xs

def _sc_gaussian_blob(h, w, cy, cx, sigma, device):
    ys, xs = _sc_grid(h, w, device)
    d2 = (ys - cy) ** 2 + (xs - cx) ** 2
    return torch.exp(-d2 / (2 * sigma ** 2))

def _sc_map_rule_of_thirds(h, w, device, focus="left"):
    points = {"left": (1/3, 1/3), "right": (1/3, 2/3),
              "bottom_left": (2/3, 1/3), "bottom_right": (2/3, 2/3)}
    cy, cx = points.get(focus, (1/3, 1/3))
    m = _sc_gaussian_blob(h, w, cy, cx, 0.18, device) * 2.0 - 0.6
    return m.clamp(-1, 1)

def _sc_map_center(h, w, device):
    m = _sc_gaussian_blob(h, w, 0.5, 0.5, 0.22, device) * 2.0 - 0.7
    return m.clamp(-1, 1)

def _sc_map_golden_spiral(h, w, device):
    ys, xs = _sc_grid(h, w, device)
    cy, cx = 0.618, 0.618
    dy, dx = ys - cy, xs - cx
    r = torch.sqrt(dy**2 + dx**2) + 1e-6
    theta = torch.atan2(dy, dx)
    b = 0.30635
    spiral = torch.cos(theta - torch.log(r) / b)
    falloff = torch.exp(-r * 2.2)
    m = spiral * falloff * 1.6
    m = m + _sc_gaussian_blob(h, w, cy, cx, 0.12, device) * 0.8 - 0.3
    return m.clamp(-1, 1)

def _sc_map_diagonal(h, w, device, direction="tl_br"):
    ys, xs = _sc_grid(h, w, device)
    d = (xs + ys) / 2 if direction == "tl_br" else (xs + (1 - ys)) / 2
    band = torch.exp(-((d - 0.5)**2) / (2 * 0.13**2))
    return (band * 2.0 - 0.8).clamp(-1, 1)

def _sc_map_horizon(h, w, device, level=1/3):
    ys, _ = _sc_grid(h, w, device)
    edge = torch.tanh((level - ys) * 10.0)
    line = torch.exp(-((ys - level)**2) / (2 * 0.02**2)) * 1.2
    return (edge * 0.5 + line).clamp(-1, 1)

def _sc_map_vignette(h, w, device):
    ys, xs = _sc_grid(h, w, device)
    d = torch.sqrt((ys - 0.5)**2 + (xs - 0.5)**2)
    return (1.0 - d * 2.6).clamp(-1, 1)

def _sc_map_leading_lines(h, w, device):
    ys, xs = _sc_grid(h, w, device)
    vy, vx = 0.33, 0.5
    m = torch.zeros(h, w, device=device)
    for sx in (0.05, 0.95):
        t_num = (xs - sx) * (vx - sx) + (ys - 1.0) * (vy - 1.0)
        t_den = (vx - sx)**2 + (vy - 1.0)**2
        t = (t_num / t_den).clamp(0, 1)
        px, py = sx + t * (vx - sx), 1.0 + t * (vy - 1.0)
        dist = torch.sqrt((xs - px)**2 + (ys - py)**2)
        m = m + torch.exp(-dist**2 / (2 * 0.035**2))
    m = m + _sc_gaussian_blob(h, w, vy, vx, 0.10, device)
    return (m * 1.4 - 0.5).clamp(-1, 1)

def _sc_map_light_gradient(h, w, device, direction="top_left"):
    ys, xs = _sc_grid(h, w, device)
    dirs = {"top_left": (1 - ys) * 0.5 + (1 - xs) * 0.5,
            "top_right": (1 - ys) * 0.5 + xs * 0.5,
            "top": (1 - ys), "left": (1 - xs)}
    d = dirs.get(direction, dirs["top_left"])
    return (d * 2.0 - 1.0).clamp(-1, 1)

_SC_COMPOSITIONS = {
    "rule of thirds (left)":      lambda h, w, d: _sc_map_rule_of_thirds(h, w, d, "left"),
    "rule of thirds (right)":     lambda h, w, d: _sc_map_rule_of_thirds(h, w, d, "right"),
    "rule of thirds (bottom-L)":  lambda h, w, d: _sc_map_rule_of_thirds(h, w, d, "bottom_left"),
    "rule of thirds (bottom-R)":  lambda h, w, d: _sc_map_rule_of_thirds(h, w, d, "bottom_right"),
    "centered subject":           lambda h, w, d: _sc_map_center(h, w, d),
    "golden spiral":              lambda h, w, d: _sc_map_golden_spiral(h, w, d),
    "diagonal (TL→BR)":           lambda h, w, d: _sc_map_diagonal(h, w, d, "tl_br"),
    "diagonal (BL→TR)":           lambda h, w, d: _sc_map_diagonal(h, w, d, "bl_tr"),
    "low horizon (big sky)":      lambda h, w, d: _sc_map_horizon(h, w, d, 2/3),
    "high horizon (big ground)":  lambda h, w, d: _sc_map_horizon(h, w, d, 1/3),
    "vignette focus":             lambda h, w, d: _sc_map_vignette(h, w, d),
    "leading lines":              lambda h, w, d: _sc_map_leading_lines(h, w, d),
    "light from top-left":        lambda h, w, d: _sc_map_light_gradient(h, w, d, "top_left"),
    "light from top-right":       lambda h, w, d: _sc_map_light_gradient(h, w, d, "top_right"),
    "light from above":           lambda h, w, d: _sc_map_light_gradient(h, w, d, "top"),
}

def _sc_lowpass_fft(img_2d, cutoff_ratio):
    h, w = img_2d.shape
    spec = torch.fft.fftshift(torch.fft.fft2(img_2d))
    cy, cx = h // 2, w // 2
    ys = torch.arange(h, device=img_2d.device).view(h, 1) - cy
    xs = torch.arange(w, device=img_2d.device).view(1, w) - cx
    radius = torch.sqrt((ys / max(cy, 1))**2 + (xs / max(cx, 1))**2)
    mask = 1.0 / (1.0 + (radius / max(cutoff_ratio, 1e-4))**6)
    return torch.fft.ifft2(torch.fft.ifftshift(spec * mask)).real

class _EmbeddedLatentCompositionGuide:
    FUNCTION = "build"
    def build(self, width, height, batch_size, composition, strength,
              frequency_cutoff, seed, channels=16, downscale=8):
        device = torch.device("cpu")
        lh, lw = height // downscale, width // downscale
        b, c = batch_size, channels
        gen = torch.Generator(device="cpu").manual_seed(seed & 0xFFFFFFFFFFFFFFFF)
        noise = torch.randn((b, c, lh, lw), generator=gen, device=device)
        comp = _SC_COMPOSITIONS[composition](lh, lw, device)
        comp = _sc_lowpass_fft(comp, frequency_cutoff)
        comp = comp / (comp.abs().max() + 1e-8)
        chan_sign = torch.tensor(
            [1.0 if i % 2 == 0 else -0.6 for i in range(c)], device=device).view(1, c, 1, 1)
        bias = comp.view(1, 1, lh, lw) * chan_sign
        shaped = noise * (1.0 - strength * 0.5) + bias * strength * 1.4
        shaped = shaped - shaped.mean(dim=(1, 2, 3), keepdim=True)
        shaped = shaped / (shaped.std(dim=(1, 2, 3), keepdim=True) + 1e-8)
        return ({"samples": shaped},)

_LatentCompositionGuide = _EmbeddedLatentCompositionGuide()
_SCENE_COMPOSITIONS = list(_SC_COMPOSITIONS.keys())

print(f"✅ SceneComposer embedded ({len(_SCENE_COMPOSITIONS)} compositions)")

# ── Register & Load PuLID ──
_pulid_model = None
_eva_clip_model = None
_face_analysis = None
_PulidApply = None

PULID_PATH = "/content/ComfyUI/custom_nodes/ComfyUI_PuLID_Flux_ll"
if os.path.exists(f"{PULID_PATH}/__init__.py"):
    print("🔄 Registering PuLID...")
    pulid_count = _load_custom_node_package(PULID_PATH, "pulid_flux_ll")
    if pulid_count:
        print(f"   → {pulid_count} nodes registered")

        # Get node classes
        _PulidModelLoader = nodes.NODE_CLASS_MAPPINGS.get("PulidFluxModelLoader")
        _PulidInsightFace = nodes.NODE_CLASS_MAPPINGS.get("PulidFluxInsightFaceLoader")
        _PulidEvaClip     = nodes.NODE_CLASS_MAPPINGS.get("PulidFluxEvaClipLoader")
        _PulidApplyCls    = nodes.NODE_CLASS_MAPPINGS.get("ApplyPulidFlux")

        if all([_PulidModelLoader, _PulidInsightFace, _PulidEvaClip, _PulidApplyCls]):
            _PulidApply = _PulidApplyCls()

            # Load PuLID model
            print("   ⏳ Loading PuLID model...")
            try:
                _pulid_model = _PulidModelLoader().load_model("pulid_flux_v0.9.1.safetensors")[0]
                print("   ✅ PuLID model loaded")
            except Exception as e:
                print(f"   ⚠️ PuLID model failed: {e}")

            # Load InsightFace
            print("   ⏳ Loading InsightFace...")
            try:
                _face_analysis = _PulidInsightFace().load_insightface("CUDA")[0]
                print("   ✅ InsightFace loaded")
            except Exception as e:
                print(f"   ⚠️ InsightFace failed: {e}")
                try:
                    _face_analysis = _PulidInsightFace().load_insightface("CPU")[0]
                    print("   ✅ InsightFace loaded (CPU fallback)")
                except Exception as e2:
                    print(f"   ⚠️ InsightFace CPU also failed: {e2}")

            # Load EVA-CLIP (auto-downloads ~1.7GB on first use)
            print("   ⏳ Loading EVA-CLIP (may download ~1.7GB first time)...")
            try:
                _eva_clip_model = _PulidEvaClip().load_eva_clip()[0]
                print("   ✅ EVA-CLIP loaded")
            except Exception as e:
                print(f"   ⚠️ EVA-CLIP failed: {e}")

            if all([_pulid_model, _eva_clip_model, _face_analysis, _PulidApply]):
                print("✅ PuLID READY — character identity available")
            else:
                print("⚠️ PuLID partially loaded — character identity may not work")
        else:
            missing = []
            if not _PulidModelLoader: missing.append("ModelLoader")
            if not _PulidInsightFace: missing.append("InsightFace")
            if not _PulidEvaClip: missing.append("EvaClip")
            if not _PulidApplyCls: missing.append("ApplyPulidFlux")
            print(f"   ⚠️ Missing nodes: {', '.join(missing)}")
    else:
        print("   ⚠️ PuLID registration failed")
else:
    print("⚠️ PuLID not found — run the install cell after Cell 1")

# ── Final summary ──
print("\n" + "=" * 50)
total = len(nodes.NODE_CLASS_MAPPINGS)
print(f"📊 Total registered nodes: {total}")
print("=" * 50)

In [ ]:
# @title Cell 2D — QwenVL Prompt

# ════════════════════════════════════════════════════════════════════
# CELL 2D — QWENVL PROMPT ENHANCEMENT
# Installs transformers + Qwen VL dependencies, defines functions
# to enhance prompts or describe reference images using Qwen2.5-VL
# Model loads on demand and offloads to CPU after use to save VRAM
# Run after Cell 2C, before Cell 3A/3B
# ════════════════════════════════════════════════════════════════════

if ENABLE_QWENVL_PROMPT:

    import os, sys, gc, subprocess

    print("📦 Installing QwenVL dependencies...")

    # Install each critical package explicitly and check results
    for pkg in ["transformers>=4.47", "accelerate", "sentencepiece",
                "qwen-vl-utils", "bitsandbytes>=0.46.1"]:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-U", "-q", pkg],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"⚠️  Failed to install {pkg}: {result.stderr.strip()[:200]}")
        else:
            print(f"   ✅ {pkg}")

    # Verify bitsandbytes is actually importable
    _bnb_available = False
    try:
        import bitsandbytes as bnb
        _bnb_available = True
        print(f"✅ bitsandbytes {bnb.__version__} verified")
    except ImportError:
        print("⚠️  bitsandbytes not available — will use float16 instead of 4-bit")
    except Exception as e:
        print(f"⚠️  bitsandbytes import error: {e} — will use float16 instead of 4-bit")

    print("✅ Dependencies installed")

    # ── Globals for QwenVL model ──
    _qwen_model = None
    _qwen_processor = None
    _qwen_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

    PROMPT_ENHANCE_SYSTEM = """You are a professional photography prompt engineer for AI image generation.
When given a short idea, expand it into a single detailed prompt paragraph.
Include: subject description, lighting (e.g. golden hour, rim lighting, studio softbox),
camera angle, lens (e.g. 85mm f/1.4), color palette, mood, background, skin/texture details,
and cinematic terms (e.g. shallow depth of field, creamy bokeh, teal-orange color grade).
Output ONLY the enhanced prompt — no explanations, no bullet points, no titles."""

    IMAGE_DESCRIBE_SYSTEM = """You are a professional image description expert for AI image generation.
Describe the provided image as a detailed prompt that could recreate it.
Include: subject, pose, expression, clothing, lighting direction and quality,
background elements, color palette, camera angle, lens characteristics,
mood, and any notable artistic or photographic techniques visible.
Output ONLY the descriptive prompt — no explanations, no bullet points, no titles."""

    def _load_qwen_model():
        """Load Qwen VL model on demand. Uses 4-bit if bitsandbytes works, else float16."""
        global _qwen_model, _qwen_processor, _bnb_available
        if _qwen_model is not None:
            return True

        import torch
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

        # ── Try 4-bit first (saves ~5 GB VRAM) ──
        if _bnb_available:
            print(f"⏳ Loading {_qwen_model_name} (4-bit quantized)...")
            try:
                from transformers import BitsAndBytesConfig
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                )
                _qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                    _qwen_model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    torch_dtype=torch.float16,
                )
                _qwen_processor = AutoProcessor.from_pretrained(_qwen_model_name)
                print("✅ QwenVL loaded (4-bit)")
                return True
            except Exception as e:
                print(f"⚠️  4-bit loading failed: {e}")
                print("   Falling back to float16...")
                _qwen_model = None
                _bnb_available = False  # don't retry 4-bit

        # ── Fallback: float16 (uses ~6-7 GB VRAM) ──
        print(f"⏳ Loading {_qwen_model_name} (float16)...")
        try:
            _qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                _qwen_model_name,
                device_map="auto",
                torch_dtype=torch.float16,
            )
            _qwen_processor = AutoProcessor.from_pretrained(_qwen_model_name)
            print("✅ QwenVL loaded (float16)")
            return True
        except Exception as e:
            print(f"❌ Failed to load QwenVL: {e}")
            _qwen_model = None
            _qwen_processor = None
            return False

    def _unload_qwen_model():
        """Unload Qwen model to free VRAM."""
        global _qwen_model, _qwen_processor
        if _qwen_model is not None:
            del _qwen_model
            _qwen_model = None
        if _qwen_processor is not None:
            del _qwen_processor
            _qwen_processor = None
        gc.collect()
        import torch
        torch.cuda.empty_cache()
        print("✅ QwenVL unloaded from VRAM")

    def enhance_prompt(text_input, auto_unload=True):
        """Expand a short idea into a detailed photographic prompt."""
        if not text_input or not text_input.strip():
            return "❌ Please enter a prompt idea to enhance."

        if not _load_qwen_model():
            return "❌ Could not load QwenVL model. Check Cell 2D output for errors."

        try:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": PROMPT_ENHANCE_SYSTEM}]},
                {"role": "user", "content": [{"type": "text", "text": text_input.strip()}]},
            ]

            text_template = _qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            inputs = _qwen_processor(
                text=[text_template],
                padding=True,
                return_tensors="pt"
            ).to(_qwen_model.device)

            import torch
            with torch.inference_mode():
                output_ids = _qwen_model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                )

            generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
            result = _qwen_processor.decode(generated_ids, skip_special_tokens=True).strip()

            if auto_unload:
                _unload_qwen_model()

            return result

        except Exception as e:
            if auto_unload:
                _unload_qwen_model()
            return f"❌ Error: {str(e)}"

    def describe_image(pil_image, auto_unload=True):
        """Describe a reference image as a detailed prompt."""
        if pil_image is None:
            return "❌ Please upload a reference image first."

        if not _load_qwen_model():
            return "❌ Could not load QwenVL model. Check Cell 2D output for errors."

        try:
            from PIL import Image as PILImage

            max_dim = 768
            w, h = pil_image.size
            if max(w, h) > max_dim:
                scale = max_dim / max(w, h)
                pil_image = pil_image.resize((int(w * scale), int(h * scale)), PILImage.LANCZOS)

            messages = [
                {"role": "system", "content": [{"type": "text", "text": IMAGE_DESCRIBE_SYSTEM}]},
                {"role": "user", "content": [
                    {"type": "image", "image": pil_image},
                    {"type": "text", "text": "Describe this image as a detailed AI generation prompt."},
                ]},
            ]

            text_template = _qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            from qwen_vl_utils import process_vision_info
            image_inputs, _ = process_vision_info(messages)

            inputs = _qwen_processor(
                text=[text_template],
                images=image_inputs,
                padding=True,
                return_tensors="pt"
            ).to(_qwen_model.device)

            import torch
            with torch.inference_mode():
                output_ids = _qwen_model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                )

            generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
            result = _qwen_processor.decode(generated_ids, skip_special_tokens=True).strip()

            if auto_unload:
                _unload_qwen_model()

            return result

        except Exception as e:
            if auto_unload:
                _unload_qwen_model()
            return f"❌ Error: {str(e)}"

    def force_unload_qwen():
        """Manual unload button."""
        _unload_qwen_model()
        return "✅ QwenVL unloaded — VRAM freed."

    print("🎉 QwenVL prompt enhancement ready!")
    mode = "4-bit" if _bnb_available else "float16"
    print(f"ℹ️  Model will load in {mode} mode on first use (~{'3' if _bnb_available else '6-7'} GB VRAM)")
    print("ℹ️  Auto-unloads after each use to free VRAM.")

else:
    # ── Stubs so Cell 3A doesn't crash ──
    def enhance_prompt(text_input, auto_unload=True):
        return "⏭️ QwenVL is disabled (ENABLE_QWENVL_PROMPT=False in Cell 1)"

    def describe_image(pil_image, auto_unload=True):
        return "⏭️ QwenVL is disabled (ENABLE_QWENVL_PROMPT=False in Cell 1)"

    def force_unload_qwen():
        return "ℹ️ QwenVL was not loaded (disabled in Cell 1)"

    print("⏭️ QwenVL prompt enhancement skipped (ENABLE_QWENVL_PROMPT=False)")
    print("   The ✍️ Prompt tab will show a disabled message.")

In [ ]:
# @title Cell 2E — Fix & Inpaint

# ════════════════════════════════════════════════════════════════════
# CELL 2E — FIX & INPAINT (CROP & STITCH)
# Auto-fix: YOLO detect → InpaintCrop → KSampler on cropped → InpaintStitch
# Uses ComfyUI-Inpaint-CropAndStitch for fast, seamless inpainting
# Run after Cell 2C, before Cell 3A
# ════════════════════════════════════════════════════════════════════

import os, sys, subprocess, random, time, math
import torch
import torch.nn.functional as TF
import numpy as np
from PIL import Image, ImageFilter, ImageDraw
from torchvision.transforms.functional import gaussian_blur as tv_gaussian_blur
from nodes import NODE_CLASS_MAPPINGS as _NCM

import comfy.sample
import comfy.samplers
import comfy.sampler_helpers

print("📦 Installing ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], capture_output=True)
print("✅ ultralytics installed")

YOLO_DIR = "/content/ComfyUI/models/yolo"
os.makedirs(YOLO_DIR, exist_ok=True)

YOLO_MODELS = {
    "face": ("face_yolov8m.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt"),
    "hand": ("hand_yolov8s.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt"),
    "person": ("person_yolov8m-seg.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/person_yolov8m-seg.pt"),
}

for name, (filename, url) in YOLO_MODELS.items():
    path = os.path.join(YOLO_DIR, filename)
    if not os.path.exists(path):
        print(f"⏳ Downloading {name} detector...")
        os.system(f'wget -q --show-progress -O "{path}" "{url}"')
    else:
        print(f"✅ {name} detector ready")

from ultralytics import YOLO
_yolo_models = {}
def _get_yolo(name):
    if name not in _yolo_models:
        _yolo_models[name] = YOLO(os.path.join(YOLO_DIR, YOLO_MODELS[name][0]))
    return _yolo_models[name]
print("✅ YOLO models ready")

def _detect_regions(pil_image, detect_types, confidence=0.3, padding_ratio=0.3):
    w, h = pil_image.size
    regions = []
    for dtype in detect_types:
        model = _get_yolo(dtype)
        results = model(pil_image, verbose=False, conf=confidence)
        for r in results:
            if r.boxes is None: continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                bw, bh = x2 - x1, y2 - y1
                pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
                x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
                x2, y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
                regions.append((dtype, x1, y1, x2, y2, float(box.conf[0])))
    return regions

# ════════════════════════════════════════════════════════════════════

@torch.inference_mode()
def _build_soft_mask(image_size, regions, blur_pixels=30):
    """Build a soft mask from YOLO regions using ellipses + gaussian blur."""
    w, h = image_size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    for (label, x1, y1, x2, y2, conf) in regions:
        # Draw filled ellipse for natural-looking mask shape
        draw.ellipse([x1, y1, x2, y2], fill=255)

    return mask

@torch.inference_mode()
def auto_fix_image(pil_image, fix_faces, fix_hands, fix_bodies,
                   fix_prompt, fix_negative, fix_denoise, fix_confidence,
                   fix_padding, fix_resolution,
                   seedvr2_enabled=False, seedvr2_resolution=1536):
    if pil_image is None: yield None, "❌ No image."; return
    detect_types = []
    if fix_faces: detect_types.append("face")
    if fix_hands: detect_types.append("hand")
    if fix_bodies: detect_types.append("person")
    if not detect_types: yield pil_image, "❌ Select a detection type."; return

    yield None, "⏳ Detecting regions..."
    regions = _detect_regions(pil_image, detect_types,
                              confidence=fix_confidence, padding_ratio=fix_padding)
    if not regions: yield pil_image, "ℹ️ No regions detected."; return

    labels_found = ", ".join(set(r[0] for r in regions))
    yield None, f"⏳ Found {len(regions)} region(s): {labels_found}"

    label_hints = {
        "face": "detailed face, clear eyes, natural skin, sharp features, symmetrical face",
        "hand": "detailed hand, five fingers, correct anatomy, natural proportions, realistic hand",
        "person": "correct anatomy, natural proportions, detailed body, realistic pose",
    }
    detected_labels = set(r[0] for r in regions)
    auto_hints = ", ".join(label_hints.get(l, "") for l in detected_labels)
    prompt = fix_prompt.strip() if fix_prompt.strip() else auto_hints
    neg = fix_negative.strip() if fix_negative.strip() else "blurry ugly bad deformed extra fingers missing fingers fused fingers mutated"

    fix_unet = active_unet

    # ── Build mask from YOLO regions ──
    mask_pil = _build_soft_mask(pil_image.size, regions, blur_pixels=0)
    mask_arr = np.array(mask_pil, dtype=np.float32) / 255.0
    mask_tensor = torch.from_numpy(mask_arr).unsqueeze(0)  # [1, H, W]

    # ── Convert image to tensor [B, H, W, C] ──
    img_arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(img_arr).unsqueeze(0)  # [1, H, W, 3]

    # ── Inpaint Crop: crop around mask, resize to target ──
    yield None, f"✂️ Cropping {len(regions)} region(s)..."
    target_res = int(fix_resolution) if int(fix_resolution) > 0 else 512
    try:
        InpaintCrop = _NCM["InpaintCropImproved"]()
    except KeyError:
        yield pil_image, "❌ InpaintCropImproved node not found. Run Cell 2C."
        return

    crop_result = InpaintCrop.inpaint_crop(
        image=img_tensor,
        downscale_algorithm="bilinear",
        upscale_algorithm="bicubic",
        preresize=False,
        preresize_mode="ensure minimum resolution",
        preresize_min_width=1024, preresize_min_height=1024,
        preresize_max_width=4096, preresize_max_height=4096,
        mask_fill_holes=True,
        mask_expand_pixels=8,
        mask_invert=False,
        mask_blend_pixels=32,
        mask_hipass_filter=0.1,
        extend_for_outpainting=False,
        extend_up_factor=1.0, extend_down_factor=1.0,
        extend_left_factor=1.0, extend_right_factor=1.0,
        context_from_mask_extend_factor=1.3,
        output_resize_to_target_size=True,
        output_target_width=target_res,
        output_target_height=target_res,
        output_padding="32",
        device_mode="gpu (much faster)",
        mask=mask_tensor,
    )
    stitcher, cropped_image, cropped_mask = crop_result[0], crop_result[1], crop_result[2]
    crop_h, crop_w = cropped_image.shape[1], cropped_image.shape[2]
    print(f"✂️ Cropped to {crop_w}x{crop_h}")

    # ── VAE Encode cropped image ──
    yield None, f"⏳ Sampling cropped region ({crop_w}x{crop_h})..."
    cropped_latent = VAEEncode.encode(vae, cropped_image.cpu())[0]

    # ── Set latent noise mask from cropped mask ──
    # cropped_mask shape: [1, H, W] → needs to be in the latent dict
    cropped_latent["noise_mask"] = cropped_mask.unsqueeze(0) if cropped_mask.dim() == 2 else cropped_mask

    # ── Encode prompts ──
    positive = CLIPTextEncode.encode(active_clip, prompt)[0]
    negative = CLIPTextEncode.encode(active_clip, neg)[0]

    # ── KSampler on cropped region ──
    denoise_val = float(fix_denoise)
    fix_steps = max(6, min(12, int(8 / max(denoise_val, 0.1))))
    seed = random.randint(0, 2**63 - 1)

    samples = KSampler.sample(fix_unet, seed, fix_steps, 1.0,
        "euler", "simple", positive, negative, cropped_latent,
        denoise=denoise_val)[0]

    # ── VAE Decode ──
    decoded = VAEDecode.decode(vae, samples)[0].detach().cpu()

    # ── Optional SeedVR2 on cropped result ──
    seedvr2_label = ""
    if seedvr2_enabled:
        yield None, f"⬆️ SeedVR2 upscale cropped region → {int(seedvr2_resolution)}px..."
        try:
            decoded_pil = _tensor_to_pil(decoded, 0)
            upscaled_pil = _run_seedvr2(decoded_pil, int(seedvr2_resolution), True, 0, 0)
            up_arr = np.array(upscaled_pil.convert("RGB"), dtype=np.float32) / 255.0
            decoded = torch.from_numpy(up_arr).unsqueeze(0)  # [1, H, W, 3]
            seedvr2_label = f" + SeedVR2→{int(seedvr2_resolution)}"
            print(f"⬆️ SeedVR2: {crop_w}x{crop_h} → {upscaled_pil.size[0]}x{upscaled_pil.size[1]}")
        except Exception as e:
            print(f"⚠️ SeedVR2 on crop failed: {e}")

    # ── Inpaint Stitch: paste back into original ──
    yield None, "✂️ Stitching back..."
    try:
        InpaintStitch = _NCM["InpaintStitchImproved"]()
    except KeyError:
        yield pil_image, "❌ InpaintStitchImproved node not found."
        return

    result_tensor = InpaintStitch.inpaint_stitch(stitcher, decoded)[0]

    # Convert back to PIL
    result = _tensor_to_pil(result_tensor, 0)

    # Resize back if needed
    orig_w, orig_h = pil_image.size
    if result.size != (orig_w, orig_h):
        result = result.resize((orig_w, orig_h), Image.LANCZOS)

    ts = int(time.time())
    result.save(f"{OUTPUT_DIR}/fixed_{ts}.png")
    yield result, f"✅ Fixed {len(regions)} ({labels_found}) — crop & stitch{seedvr2_label}"

# ════════════════════════════════════════════════════════════════════
# MANUAL INPAINT — uses CropAndStitch for fast, seamless results
# ════════════════════════════════════════════════════════════════════

@torch.inference_mode()
def manual_inpaint(editor_data, inpaint_prompt, inpaint_negative, inpaint_denoise, inpaint_steps, inpaint_resolution=1024, seedvr2_enabled=False, seedvr2_resolution=1536):
    if editor_data is None: yield None, "❌ No image."; return

    if isinstance(editor_data, dict):
        bg = editor_data.get("background")
        layers = editor_data.get("layers", [])
        composite = editor_data.get("composite")
        if bg is None and composite is not None:
            pil_image = composite.convert("RGB") if isinstance(composite, Image.Image) else None
        elif bg is not None:
            pil_image = bg.convert("RGB") if isinstance(bg, Image.Image) else None
        else:
            yield None, "❌ Could not read image."; return

        if layers and len(layers) > 0:
            mask_combined = Image.new("L", pil_image.size, 0)
            for layer in layers:
                if isinstance(layer, Image.Image) and layer.mode == "RGBA":
                    alpha = layer.split()[3]
                    mask_combined = Image.composite(Image.new("L", pil_image.size, 255), mask_combined, alpha)
            mask_pil = mask_combined
        else:
            yield None, "❌ No mask painted."; return
    else:
        yield None, "❌ Paint a mask first."; return

    mask_arr = np.array(mask_pil)
    if mask_arr.max() < 10: yield None, "❌ No mask painted."; return

    prompt = inpaint_prompt.strip() or "high quality, detailed, natural"
    neg = inpaint_negative.strip() or "blurry ugly bad"

    # Convert to tensors
    img_arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(img_arr).unsqueeze(0)  # [1, H, W, 3]
    mask_tensor = torch.from_numpy(np.array(mask_pil, dtype=np.float32) / 255.0).unsqueeze(0)  # [1, H, W]

    # ── InpaintCrop ──
    yield None, "✂️ Cropping masked region..."
    try:
        InpaintCrop = _NCM["InpaintCropImproved"]()
    except KeyError:
        yield None, "❌ InpaintCropImproved not found. Run Cell 2C."; return

    crop_result = InpaintCrop.inpaint_crop(
        image=img_tensor,
        downscale_algorithm="bilinear",
        upscale_algorithm="bicubic",
        preresize=False,
        preresize_mode="ensure minimum resolution",
        preresize_min_width=1024, preresize_min_height=1024,
        preresize_max_width=4096, preresize_max_height=4096,
        mask_fill_holes=True,
        mask_expand_pixels=4,
        mask_invert=False,
        mask_blend_pixels=32,
        mask_hipass_filter=0.1,
        extend_for_outpainting=False,
        extend_up_factor=1.0, extend_down_factor=1.0,
        extend_left_factor=1.0, extend_right_factor=1.0,
        context_from_mask_extend_factor=1.5,
        output_resize_to_target_size=True,
        output_target_width=int(inpaint_resolution),
        output_target_height=int(inpaint_resolution),
        output_padding="32",
        device_mode="gpu (much faster)",
        mask=mask_tensor,
    )
    stitcher, cropped_image, cropped_mask = crop_result[0], crop_result[1], crop_result[2]
    crop_h, crop_w = cropped_image.shape[1], cropped_image.shape[2]
    print(f"✂️ Cropped to {crop_w}x{crop_h}")

    # ── VAE Encode ──
    yield None, f"⏳ Inpainting cropped region ({crop_w}x{crop_h})..."
    cropped_latent = VAEEncode.encode(vae, cropped_image.cpu())[0]
    cropped_latent["noise_mask"] = cropped_mask.unsqueeze(0) if cropped_mask.dim() == 2 else cropped_mask

    # ── Encode prompts ──
    positive = CLIPTextEncode.encode(active_clip, prompt)[0]
    negative = CLIPTextEncode.encode(active_clip, neg)[0]

    # ── KSampler on cropped region ──
    denoise_val = float(inpaint_denoise)
    steps = int(inpaint_steps)
    seed = random.randint(0, 2**63 - 1)

    samples = KSampler.sample(active_unet, seed, steps, 1.0,
        "euler", "simple", positive, negative, cropped_latent,
        denoise=denoise_val)[0]

    # ── VAE Decode ──
    decoded = VAEDecode.decode(vae, samples)[0].detach().cpu()

    # ── InpaintStitch ──
    yield None, "✂️ Stitching back..."
    try:
        InpaintStitch = _NCM["InpaintStitchImproved"]()
    except KeyError:
        yield None, "❌ InpaintStitchImproved not found."; return

    result_tensor = InpaintStitch.inpaint_stitch(stitcher, decoded)[0]
    result = _tensor_to_pil(result_tensor, 0)

    w, h = pil_image.size
    if result.size != (w, h):
        result = result.resize((w, h), Image.LANCZOS)

    ts = int(time.time())
    result.save(f"{OUTPUT_DIR}/inpainted_{ts}.png")
    yield result, f"✅ Inpainted (crop & stitch, {crop_w}x{crop_h})"




# ════════════════════════════════════════════════════════════════════
# POSE EXTRACTION & EDITOR (for Qwen Edit pose transfer)
# ════════════════════════════════════════════════════════════════════

_JOINT_NAMES = [
    "Nose", "Neck", "R.Shoulder", "R.Elbow", "R.Wrist",
    "L.Shoulder", "L.Elbow", "L.Wrist",
    "R.Hip", "R.Knee", "R.Ankle", "L.Hip", "L.Knee", "L.Ankle",
    "R.Eye", "L.Eye", "R.Ear", "L.Ear",
]

def _dwpose_extract(pil_image):
    """Run DWPose. Returns (skeleton_pil, openpose_dict) or (None, None)."""
    if _DWPose is None: return None, None
    arr = np.array(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    t = torch.from_numpy(arr).unsqueeze(0)
    res = max(pil_image.size)
    res = min(1024, (res // 64) * 64)
    out = _DWPose.estimate_pose(t, detect_hand="enable", detect_body="enable",
                                detect_face="disable", resolution=res)
    if isinstance(out, dict):
        img_t, dicts = out["result"]
    else:
        img_t, dicts = out
    skel = _tensor_to_pil(img_t, 0).resize(pil_image.size, Image.NEAREST)
    d = dicts[0] if dicts else None
    return skel, d

def _kp_xy(flat, idx):
    if flat is None or idx * 3 + 2 >= len(flat): return None
    x, y, c = flat[idx*3], flat[idx*3+1], flat[idx*3+2]
    return (x, y, c) if c > 0 else None

def _pose_overlay(image_pil, skel_pil, alpha=0.5):
    img = image_pil.convert("RGBA")
    sk = skel_pil.convert("RGBA")
    sk_arr = np.array(sk)
    mask = (sk_arr[:,:,0] > 10) | (sk_arr[:,:,1] > 10) | (sk_arr[:,:,2] > 10)
    sk_arr[:,:,3] = np.where(mask, int(alpha * 255), 0)
    sk_rgba = Image.fromarray(sk_arr)
    img.paste(sk_rgba, (0, 0), sk_rgba)
    return img.convert("RGB")

def _render_skeleton_direct(flat_kps, w, h):
    """Render skeleton directly from flat [x,y,c,...] normalized keypoints."""
    import PIL.ImageDraw as _Draw
    LIMBS = [
        (1,2,(255,0,0)),(2,3,(255,85,0)),(3,4,(255,170,0)),
        (1,5,(0,255,0)),(5,6,(0,204,68)),(6,7,(0,170,136)),
        (1,0,(0,0,255)),
        (0,14,(255,0,255)),(14,16,(255,0,204)),(0,15,(0,255,255)),(15,17,(0,204,255)),
        (1,8,(255,255,0)),(8,9,(204,204,0)),(9,10,(170,170,0)),
        (1,11,(255,0,255)),(11,12,(204,0,204)),(12,13,(170,0,170)),
    ]
    kps = []
    for i in range(0, len(flat_kps), 3):
        kps.append((flat_kps[i], flat_kps[i+1], flat_kps[i+2]))
    canvas = Image.new("RGB", (w, h), (0, 0, 0))
    draw = _Draw.Draw(canvas)
    for a, b, color in LIMBS:
        if a >= len(kps) or b >= len(kps): continue
        ka, kb = kps[a], kps[b]
        if ka[2] < 0.1 or kb[2] < 0.1: continue
        x1, y1 = int(ka[0] * w), int(ka[1] * h)
        x2, y2 = int(kb[0] * w), int(kb[1] * h)
        draw.line([(x1, y1), (x2, y2)], fill=color, width=4)
    for i, (x, y, c) in enumerate(kps[:18]):
        if c < 0.1: continue
        px, py = int(x * w), int(y * h)
        r = 5
        draw.ellipse([px-r, py-r, px+r, py+r], fill=(255, 255, 255), outline=(0, 0, 0))
    return canvas

@torch.inference_mode()
def extract_pose(pil_image):
    if pil_image is None: return None, None, None, None, "❌ No image."
    if _DWPose is None: return None, None, None, None, "❌ DWPose not loaded. Run Cell 2C."
    skel, pdict = _dwpose_extract(pil_image)
    if pdict is None or not pdict.get("people"):
        return None, None, None, None, "❌ No person detected."
    overlay = _pose_overlay(pil_image, skel, alpha=0.6)
    body = pdict["people"][0].get("pose_keypoints_2d", [])
    w, h = pil_image.size
    info_lines = []
    for i, name in enumerate(_JOINT_NAMES):
        kp = _kp_xy(body, i)
        if kp:
            px = kp[0] * w if kp[0] <= 1.5 else kp[0]
            py = kp[1] * h if kp[1] <= 1.5 else kp[1]
            info_lines.append(f"  {i:2d} {name}: ({int(px)}, {int(py)})")
    return pil_image, overlay, skel, pdict, f"✅ Pose extracted — {len(info_lines)} joints"

def apply_edited_pose(pil_image, pose_dict, pose_json_str):
    if not pose_json_str or not pose_json_str.strip():
        return None, None, None, "❌ No pose data — drag joints first."
    import json as _j
    try: flat = _j.loads(pose_json_str)
    except: return None, None, None, "❌ Invalid pose JSON."
    if pose_dict is None or not pose_dict.get("people"):
        pose_dict = {"people": [{"pose_keypoints_2d": [], "hand_left_keypoints_2d": [], "hand_right_keypoints_2d": [], "face_keypoints_2d": []}], "canvas_width": pil_image.size[0], "canvas_height": pil_image.size[1]}
    new_dict = _j.loads(_j.dumps(pose_dict))
    new_dict["people"][0]["pose_keypoints_2d"] = flat
    w, h = pil_image.size
    try:
        new_skel = _render_skeleton_direct(flat, w, h)
        print(f"✅ Skeleton rendered ({w}x{h}), kp range: x=[{min(flat[::3]):.3f}-{max(flat[::3]):.3f}] y=[{min(flat[1::3]):.3f}-{max(flat[1::3]):.3f}]")
    except Exception as e:
        return None, None, None, f"❌ Render failed: {e}"
    overlay = _pose_overlay(pil_image, new_skel, 0.6)
    return new_skel, overlay, new_dict, "✅ Pose applied — ready for Qwen Edit"

def _save_pose_editor_html(image_pil, pose_dict):
    """Save interactive pose editor as HTML file."""
    import base64
    from io import BytesIO
    w, h = image_pil.size
    scale = min(800 / w, 650 / h, 1.0)
    cw, ch = int(w * scale), int(h * scale)
    buf = BytesIO()
    image_pil.save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    body = pose_dict["people"][0] if pose_dict and pose_dict.get("people") else {}
    kps_raw = list(body.get("pose_keypoints_2d", []))
    if kps_raw:
        xs = [kps_raw[i] for i in range(0, len(kps_raw), 3)]
        ys = [kps_raw[i] for i in range(1, len(kps_raw), 3)]
        if max(xs + ys, default=0) > 1.5:
            for i in range(0, len(kps_raw), 3):
                kps_raw[i] = kps_raw[i] / w
                kps_raw[i+1] = kps_raw[i+1] / h
    kps_str = ",".join(str(round(x, 6)) for x in kps_raw)
    html = (
        '<!DOCTYPE html><html><head><meta charset="utf-8"><title>Pose Editor</title><style>'
        'body{margin:0;padding:10px;background:#111;color:#fff;font-family:sans-serif;}'
        'canvas{display:block;cursor:crosshair;margin:0 auto;border:2px solid #555;border-radius:8px;}'
        '#info{color:#0f0;font:13px monospace;text-align:center;padding:6px;}'
        '#json-area{width:100%;max-width:' + str(cw) + 'px;height:60px;margin:8px auto;display:block;'
        'background:#222;color:#0f0;border:1px solid #555;font:11px monospace;border-radius:4px;padding:4px;}'
        '#copy-btn{display:block;margin:8px auto;padding:10px 30px;font-size:16px;font-weight:bold;'
        'background:#f80;color:#000;border:none;border-radius:8px;cursor:pointer;}'
        '#copy-btn:hover{background:#fa0;}'
        'h2{text-align:center;color:#f80;margin:5px 0;}'
        'p.hint{text-align:center;color:#888;font-size:13px;margin:2px;}'
        '</style></head><body>'
        '<h2>Pose Editor</h2>'
        '<p class="hint">Drag joint circles. Wrists/ankles auto-solve elbow/knee.</p>'
        '<canvas id="c" width="' + str(cw) + '" height="' + str(ch) + '"></canvas>'
        '<div id="info">Ready</div>'
        '<button id="copy-btn" onclick="copyJSON()">Copy Pose JSON</button>'
        '<textarea id="json-area" readonly></textarea>'
        '<p class="hint">After editing, click Copy, go back to Gradio, paste in the JSON box, click Apply.</p>'
        '<script>'
        'var W=' + str(cw) + ',H=' + str(ch) + ',oW=' + str(w) + ',oH=' + str(h) + ',R=9,dr=-1,hv=-1;'
        'var LM=[[1,2,"#f00"],[2,3,"#f50"],[3,4,"#fa0"],[1,5,"#0f0"],[5,6,"#0c4"],[6,7,"#0a8"],'
        '[1,0,"#00f"],[0,14,"#f0f"],[14,16,"#f0c"],[0,15,"#0ff"],[15,17,"#0cf"],'
        '[1,8,"#ff0"],[8,9,"#cc0"],[9,10,"#aa0"],[1,11,"#f0f"],[11,12,"#c0c"],[12,13,"#a0a"]];'
        'var NM=["Nose","Neck","R.Sho","R.Elb","R.Wri","L.Sho","L.Elb","L.Wri",'
        '"R.Hip","R.Kne","R.Ank","L.Hip","L.Kne","L.Ank","R.Eye","L.Eye","R.Ear","L.Ear"];'
        'var IK={4:[2,3,4],7:[5,6,7],10:[8,9,10],13:[11,12,13]};'
        'var raw=[' + kps_str + '];'
        'var kp=[];for(var i=0;i<raw.length;i+=3)kp.push([raw[i],raw[i+1],raw[i+2]]);'
        'var c=document.getElementById("c"),ctx=c.getContext("2d"),inf=document.getElementById("info"),ja=document.getElementById("json-area");'
        'var img=new Image();img.src="data:image/jpeg;base64,' + b64 + '";'
        'function sIK(ri,mi,ei,tx,ty){'
        'var r=kp[ri],m=kp[mi],e=kp[ei],asp=oW/oH;if(!r||r[2]<.1)return;'
        'function D(a,b){var dx=(a[0]-b[0])*asp,dy=a[1]-b[1];return Math.sqrt(dx*dx+dy*dy);}'
        'var uL=m&&m[2]>.1?D(r,m):D(r,[tx,ty,1])/2,fL=m&&m[2]>.1&&e&&e[2]>.1?D(m,e):uL;'
        'var bs=1;if(m&&m[2]>.1&&e&&e[2]>.1){var v1x=(m[0]-r[0])*asp,v1y=m[1]-r[1],v2x=(e[0]-r[0])*asp,v2y=e[1]-r[1];'
        'bs=v1x*v2y-v1y*v2x>=0?1:-1;}else bs=ei==4||ei==10?1:-1;'
        'var rx=r[0]*asp,ry=r[1],wx=tx*asp,wy=ty,dx=wx-rx,dy=wy-ry,dd=Math.sqrt(dx*dx+dy*dy);'
        'var rc=uL+fL;if(dd>rc*.98)dd=rc*.98;if(dd<1e-6)return;'
        'var ux=dx/dd,uy=dy/dd,a2=(uL*uL-fL*fL+dd*dd)/(2*dd),h=Math.sqrt(Math.max(0,uL*uL-a2*a2));'
        'var px=rx+a2*ux,py=ry+a2*uy;kp[mi]=[(px+bs*h*(-uy))/asp,py+bs*h*ux,1];kp[ei]=[tx,ty,1];}'
        'function draw(){'
        'ctx.clearRect(0,0,W,H);if(img.complete)ctx.drawImage(img,0,0,W,H);ctx.lineWidth=3;'
        'LM.forEach(function(l){var a=l[0],b=l[1];if(a>=kp.length||b>=kp.length)return;'
        'var ka=kp[a],kb=kp[b];if(!ka||ka[2]<.1||!kb||kb[2]<.1)return;'
        'ctx.strokeStyle=l[2];ctx.beginPath();ctx.moveTo(ka[0]*W,ka[1]*H);ctx.lineTo(kb[0]*W,kb[1]*H);ctx.stroke();});'
        'for(var i=0;i<Math.min(kp.length,18);i++){var k=kp[i];if(!k||k[2]<.1)continue;'
        'var x=k[0]*W,y=k[1]*H;ctx.fillStyle=i===dr?"#ff0":i===hv?"#0ff":"#fff";'
        'ctx.strokeStyle="#000";ctx.lineWidth=2;ctx.beginPath();ctx.arc(x,y,R,0,Math.PI*2);ctx.fill();ctx.stroke();'
        'ctx.fillStyle="#ff0";ctx.font="bold 12px sans-serif";ctx.fillText(NM[i]||i,x+R+2,y-R);}'
        'var f=[];kp.forEach(function(k){f.push(k[0],k[1],k[2]);});ja.value=JSON.stringify(f);}'
        'img.onload=draw;'
        'function fj(mx,my){for(var i=0;i<Math.min(kp.length,18);i++){var k=kp[i];if(!k||k[2]<.1)continue;'
        'var dx=mx-k[0]*W,dy=my-k[1]*H;if(dx*dx+dy*dy<(R+6)*(R+6))return i;}return-1;}'
        'c.onmousedown=function(e){var r=c.getBoundingClientRect();dr=fj(e.clientX-r.left,e.clientY-r.top);};'
        'c.onmousemove=function(e){var r=c.getBoundingClientRect(),mx=e.clientX-r.left,my=e.clientY-r.top;'
        'if(dr>=0){var tx=mx/W,ty=my/H;if(IK[dr]){var ch=IK[dr];sIK(ch[0],ch[1],ch[2],tx,ty);}'
        'else kp[dr]=[tx,ty,1];draw();inf.textContent=NM[dr]+": ("+Math.round(tx*oW)+","+Math.round(ty*oH)+")";}'
        'else{var h2=fj(mx,my);if(h2!==hv){hv=h2;draw();}if(h2>=0)inf.textContent="Hover: "+NM[h2];}};'
        'c.onmouseup=function(){if(dr>=0)inf.textContent="Moved "+NM[dr]+". Drag more or Copy.";dr=-1;};'
        'c.onmouseleave=function(){dr=-1;hv=-1;draw();};'
        'c.ontouchstart=function(e){e.preventDefault();var t=e.touches[0],r=c.getBoundingClientRect();dr=fj(t.clientX-r.left,t.clientY-r.top);};'
        'c.ontouchmove=function(e){e.preventDefault();if(dr<0)return;var t=e.touches[0],r=c.getBoundingClientRect();'
        'var tx=(t.clientX-r.left)/W,ty=(t.clientY-r.top)/H;'
        'if(IK[dr]){var ch=IK[dr];sIK(ch[0],ch[1],ch[2],tx,ty);}else kp[dr]=[tx,ty,1];draw();};'
        'c.ontouchend=function(){dr=-1;};'
        'function copyJSON(){var f=[];kp.forEach(function(k){f.push(k[0],k[1],k[2]);});'
        'var j=JSON.stringify(f);navigator.clipboard.writeText(j).then(function(){'
        'document.getElementById("copy-btn").textContent="Copied!";'
        'setTimeout(function(){document.getElementById("copy-btn").textContent="Copy Pose JSON";},2000);'
        '}).catch(function(){ja.select();document.execCommand("copy");'
        'document.getElementById("copy-btn").textContent="Copied!";});}'
        '</script></body></html>'
    )
    path = OUTPUT_DIR + "/pose_editor.html"
    with open(path, "w") as f:
        f.write(html)
    return path


print("🎉 Fix, Inpaint & Pose Edit ready (Crop & Stitch mode)!")

In [ ]:
# @title Cell 2F — Dual CFG

# ════════════════════════════════════════════════════════════════════
# CELL 2F — DUAL CFG GENERATOR
# Two-pass KSamplerAdvanced: low CFG for variation → high CFG for adherence
# Based on community workflow: split steps with different CFG values
# Run after Cell 2C, before Cell 3A/3B
# ════════════════════════════════════════════════════════════════════

import torch, random, time
from nodes import NODE_CLASS_MAPPINGS as _NCM

KSamplerAdvanced = _NCM.get("KSamplerAdvanced")
if KSamplerAdvanced:
    KSamplerAdvanced = KSamplerAdvanced()
    print("✅ KSamplerAdvanced loaded")
else:
    raise RuntimeError("KSamplerAdvanced not found in NODE_CLASS_MAPPINGS")

@torch.inference_mode()
def _generate_dual_cfg(positive_prompt, negative_prompt, width, height,
                       batch_size, seed, total_steps, cfg_low, cfg_high,
                       split_step, sampler_name, scheduler,
                       flow_dpo_enabled=False, flow_dpo_strength=0.8,
                       comp_enabled=False, comp_type="centered subject"):
    """
    Two-pass split-CFG generation:
      Pass 1: steps 0 → split_step with low CFG (adds variation/creativity)
      Pass 2: steps split_step → end with high CFG (enforces prompt adherence)
    """
    global last_generated_pil
    reset_stop()
    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    gen_unet = active_unet; gen_clip = active_clip
    if flow_dpo_enabled:
        yield None, int(seed), "⏳ Applying Flow-DPO..."
        gen_unet, gen_clip = _apply_flow_dpo(active_unet, active_clip, flow_dpo_strength)

    comp_label = ""
    final_prompt = positive_prompt
    if comp_enabled:
        comp_label = f" + 🎯 {comp_type}"
        final_prompt = _apply_composition_to_prompt(positive_prompt, comp_type)

    positive = CLIPTextEncode.encode(gen_clip, final_prompt)[0]
    negative = CLIPTextEncode.encode(gen_clip, negative_prompt)[0]
    latent = EmptyLatentImage.generate(int(width), int(height), batch_size=int(batch_size))[0]

    split = int(split_step)
    steps = int(total_steps)

    # ── Pass 1: low CFG → variation ──
    yield None, int(seed), f"⏳ Pass 1/{2}: CFG {cfg_low} (variation, steps 0→{split})..."
    pass1 = KSamplerAdvanced.sample(
        gen_unet, "enable", int(seed), steps, float(cfg_low),
        sampler_name, scheduler, positive, negative, latent,
        start_at_step=0, end_at_step=split,
        return_with_leftover_noise="disable"
    )[0]

    if stop_flag:
        yield None, int(seed), "⏹️ Stopped after pass 1"
        return

    # ── Pass 2: high CFG → prompt adherence ──
    yield None, int(seed), f"⏳ Pass 2/{2}: CFG {cfg_high} (adherence, steps {split}→{steps})..."
    pass2 = KSamplerAdvanced.sample(
        gen_unet, "enable", int(seed), steps, float(cfg_high),
        sampler_name, scheduler, positive, negative, pass1,
        start_at_step=split, end_at_step=10000,
        return_with_leftover_noise="disable"
    )[0]

    # ── Decode ──
    decoded = VAEDecode.decode(vae, pass2)[0].detach()
    images = []; ts = int(time.time())
    for i in range(decoded.shape[0]):
        img = _tensor_to_pil(decoded, i)
        img.save(f"{OUTPUT_DIR}/z_dualcfg_{ts}_{i}.png")
        images.append(img)

    if images:
        last_generated_pil = images[0]

    label = f"✅ Dual CFG ({cfg_low}→{cfg_high}, split@{split}/{steps})"
    if flow_dpo_enabled: label += " + Flow-DPO"
    label += comp_label
    yield images, int(seed), label

print("🎉 Dual CFG generator ready!")


In [ ]:
# @title Cell 2G — Qwen Edit Pose

# ════════════════════════════════════════════════════════════════════
# CELL 2G — QWEN IMAGE EDIT 2511 (POSE TRANSFER)
# Instruction-following editor: "put this person in that pose"
# Models load lazily on first use to save VRAM.
# Run after Cell 2C. Requires DOWNLOAD_QWEN_EDIT=True in Cell 1.
# ════════════════════════════════════════════════════════════════════

import os, time, random, gc
import torch
import numpy as np
from PIL import Image
import nodes as _nodes
import comfy.model_management as _mm

_QE_MODEL_NAME = "qwen-image-edit-2511-Q4_K_M.gguf"
_QE_CLIP_NAME  = "qwen_2.5_vl_7b_fp8_scaled.safetensors"
_QE_VAE_NAME   = "qwen_image_vae.safetensors"
_QE_LORA_NAME  = "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"

_qe_available = all([
    os.path.exists(f"/content/ComfyUI/models/diffusion_models/{_QE_MODEL_NAME}"),
    os.path.exists(f"/content/ComfyUI/models/text_encoders/{_QE_CLIP_NAME}"),
    os.path.exists(f"/content/ComfyUI/models/vae/{_QE_VAE_NAME}"),
])
_qe_lora_available = os.path.exists(f"/content/ComfyUI/models/loras/{_QE_LORA_NAME}")

# Node classes
_UnetLoaderGGUF = _nodes.NODE_CLASS_MAPPINGS.get("UnetLoaderGGUF")
_TextEncodeQwenEditPlus = _nodes.NODE_CLASS_MAPPINGS.get("TextEncodeQwenImageEditPlus")
_FluxKontextRefMethod = _nodes.NODE_CLASS_MAPPINGS.get("FluxKontextMultiReferenceLatentMethod")
_CFGNorm = _nodes.NODE_CLASS_MAPPINGS.get("CFGNorm")
_ModelSamplingAuraFlow_cls = _nodes.NODE_CLASS_MAPPINGS.get("ModelSamplingAuraFlow")
_LoraLoaderModelOnly = _nodes.NODE_CLASS_MAPPINGS.get("LoraLoaderModelOnly")

_qe_missing = [n for n, c in [("UnetLoaderGGUF", _UnetLoaderGGUF), ("TextEncodeQwenImageEditPlus", _TextEncodeQwenEditPlus),
               ("FluxKontextMultiReferenceLatentMethod", _FluxKontextRefMethod), ("CFGNorm", _CFGNorm),
               ("ModelSamplingAuraFlow", _ModelSamplingAuraFlow_cls)] if c is None]

if _qe_missing:
    print(f"⚠️ Qwen Edit: missing nodes {_qe_missing} — run Cell 2C / update ComfyUI")
    _qe_available = False
elif _qe_available:
    print(f"✅ Qwen Edit models found | Lightning LoRA: {'yes' if _qe_lora_available else 'no'}")
else:
    print("⚠️ Qwen Edit models not found — set DOWNLOAD_QWEN_EDIT=True in Cell 1")

# Lazy-loaded model cache
_qe_cache = {"unet": None, "clip": None, "vae": None, "unet_lora": None}

def _qe_load_models(use_lightning=True):
    """Load Qwen Edit models on first use. Unloads Z-Image models first to free VRAM."""
    if _qe_cache["unet"] is not None and _qe_cache["clip"] is not None and _qe_cache["vae"] is not None:
        return True
    print("🔄 Loading Qwen Edit models (first use — unloading Z-Image to free VRAM)...")
    _mm.unload_all_models()
    _mm.soft_empty_cache()
    gc.collect(); torch.cuda.empty_cache()

    _qe_cache["unet"] = _UnetLoaderGGUF().load_unet(_QE_MODEL_NAME)[0]
    _qe_cache["clip"] = CLIPLoader.load_clip(_QE_CLIP_NAME, "qwen_image", "default")[0]
    _qe_cache["vae"]  = VAELoader.load_vae(_QE_VAE_NAME)[0]

    if use_lightning and _qe_lora_available and _LoraLoaderModelOnly is not None:
        _qe_cache["unet_lora"] = _LoraLoaderModelOnly().load_lora_model_only(
            _qe_cache["unet"], _QE_LORA_NAME, 1.0)[0]
        print("✅ Qwen Edit + Lightning 4-step LoRA loaded")
    else:
        _qe_cache["unet_lora"] = _qe_cache["unet"]
        print("✅ Qwen Edit loaded (no Lightning — use 8+ steps)")
    return True

def _qe_unload():
    """Free Qwen Edit models."""
    for k in _qe_cache: _qe_cache[k] = None
    _mm.unload_all_models(); _mm.soft_empty_cache()
    gc.collect(); torch.cuda.empty_cache()
    print("🗑️ Qwen Edit unloaded")

def _pil_to_tensor(pil):
    arr = np.array(pil.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0)

@torch.inference_mode()
def qwen_pose_transfer(portrait_pil, skeleton_pil, prompt_extra, negative_prompt,
                       steps, cfg, seed, use_lightning, target_mp, unload_after):
    """
    Pose transfer with Qwen-Image-Edit-2511.
      image1 = pose skeleton, image2 = portrait
      prompt: 'Image 1 is the pose reference. Make the person in image 2 copy that exact pose. ...'
    """
    # Detailed diagnostics
    _m = f"/content/ComfyUI/models/diffusion_models/{_QE_MODEL_NAME}"
    _c = f"/content/ComfyUI/models/text_encoders/{_QE_CLIP_NAME}"
    _v = f"/content/ComfyUI/models/vae/{_QE_VAE_NAME}"
    diag = []
    diag.append(f"model: {'OK' if os.path.exists(_m) else 'MISSING'} {_m}")
    diag.append(f"clip:  {'OK' if os.path.exists(_c) else 'MISSING'} {_c}")
    diag.append(f"vae:   {'OK' if os.path.exists(_v) else 'MISSING'} {_v}")
    diag.append(f"nodes missing: {_qe_missing if _qe_missing else 'none'}")
    print("🎭 Qwen Edit diagnostics:\n  " + "\n  ".join(diag))

    if not _qe_available:
        msg = "❌ Qwen Edit not available:\n" + "\n".join(diag) + "\n\nRe-run Cell 1 (with DOWNLOAD_QWEN_EDIT=True, ~22GB) then Cell 2G."
        print(msg)
        yield None, msg; return
    if portrait_pil is None:
        print("❌ No portrait"); yield None, "❌ No portrait."; return
    if skeleton_pil is None:
        print("❌ No skeleton"); yield None, "❌ No skeleton — click Apply first."; return
    print(f"🎭 Starting Qwen pose transfer: portrait {portrait_pil.size}, skeleton {skeleton_pil.size}")

    yield None, "🔄 Loading Qwen Edit (first time ~30s)..."
    try:
        _qe_load_models(use_lightning=bool(use_lightning))
    except Exception as e:
        yield None, f"❌ Load failed: {e}"; return

    unet = _qe_cache["unet_lora"] if use_lightning else _qe_cache["unet"]
    clip = _qe_cache["clip"]; qvae = _qe_cache["vae"]

    # Resize portrait to target megapixels (Qwen works best ~1MP)
    w, h = portrait_pil.size
    mp = float(target_mp)
    scale = (mp * 1_000_000 / (w * h)) ** 0.5
    nw, nh = int(round(w * scale / 16)) * 16, int(round(h * scale / 16)) * 16
    portrait_r = portrait_pil.resize((nw, nh), Image.LANCZOS)
    skeleton_r = skeleton_pil.resize((nw, nh), Image.LANCZOS)

    # Model patches (matches reference workflow)
    yield None, "⚙️ Patching model (AuraFlow shift 3, CFGNorm)..."
    m = _ModelSamplingAuraFlow_cls().patch_aura(unet, 3.0)[0]
    m = _CFGNorm.execute(m, 1.0).args[0]

    # Prompt — image1 = skeleton, image2 = portrait (matches reference workflow order)
    base = ("Image 1 is a pose skeleton reference. Make the person in image 2 adopt that exact body pose "
            "and limb positions. Keep the same face, hair, clothes, and background as image 2. "
            "Only change the pose.")
    extra = (prompt_extra or "").strip()
    full_prompt = base + (" " + extra if extra else "")
    neg = (negative_prompt or "").strip() or "blurry, deformed, extra limbs, different person, changed clothes"

    yield None, "📝 Encoding (VL encoder sees both images)..."
    t_skel = _pil_to_tensor(skeleton_r)
    t_port = _pil_to_tensor(portrait_r)
    pos = _TextEncodeQwenEditPlus.execute(clip, full_prompt, vae=qvae, image1=t_skel, image2=t_port).args[0]
    negc = _TextEncodeQwenEditPlus.execute(clip, neg, vae=qvae, image1=t_skel, image2=t_port).args[0]
    pos = _FluxKontextRefMethod.execute(pos, "index_timestep_zero").args[0]
    negc = _FluxKontextRefMethod.execute(negc, "index_timestep_zero").args[0]

    # Start latent from the portrait (denoise 1.0 — reference conditioning does the rest)
    latent = VAEEncode.encode(qvae, t_port)[0]

    sd = int(seed) if int(seed) > 0 else random.randint(1, 2**31)
    st = int(steps) if not use_lightning else min(int(steps), 4)
    yield None, f"⏳ Generating (steps {st}, cfg {cfg}, {nw}x{nh})..."
    samples = KSampler.sample(m, sd, st, float(cfg), "euler", "simple", pos, negc, latent, denoise=1.0)[0]

    decoded = VAEDecode.decode(qvae, samples)[0].detach()
    result = _tensor_to_pil(decoded, 0)
    if result.size != (w, h):
        result = result.resize((w, h), Image.LANCZOS)

    ts = int(time.time())
    result.save(f"{OUTPUT_DIR}/qwen_pose_{ts}.png")
    skeleton_pil.save(f"{OUTPUT_DIR}/qwen_pose_skel_{ts}.png")

    if unload_after:
        _qe_unload()

    yield result, f"✅ Pose transferred (Qwen Edit 2511, {st} steps, seed {sd})"

print("🎉 Qwen Edit pose transfer ready" if _qe_available else "⚠️ Qwen Edit unavailable")


In [ ]:
# @title Cell 3A — Gradio UI

# ════════════════════════════════════════════════════════════════════
# CELL 3A — GRADIO SHARE (PRIMARY LAUNCHER)
# V4 LoRA, refine LoRA, auto color, composition, ANC, Z-Sampler Turbo
# Compare slider, CapitanZiT, civitai, download file
# ════════════════════════════════════════════════════════════════════

import gradio as gr
import glob, os, time, re, subprocess, urllib.parse, base64, io

SAMPLERS = ["euler", "euler_ancestral", "heun", "dpm_2", "dpm_2_ancestral",
            "lms", "dpmpp_2s_ancestral", "dpmpp_sde", "dpmpp_2m",
            "dpmpp_2m_sde", "dpmpp_3m_sde", "ddim", "uni_pc",
            "minimal_change_flow"]
SCHEDULERS = ["simple", "normal", "karras", "exponential", "sgm_uniform",
              "ddim_uniform", "beta", "capitanZiT", "smooth_cosine"]
COLOR_METHODS = ["wavelet", "lab", "wavelet_adaptive", "hsv", "adain", "none"]
COMP_CHOICES = list(_COMP_PROMPTS.keys()) if _COMP_PROMPTS else [
    "rule of thirds (left)", "rule of thirds (right)",
    "rule of thirds (bottom-L)", "rule of thirds (bottom-R)",
    "centered subject", "golden spiral",
    "diagonal (TL→BR)", "diagonal (BL→TR)",
    "low horizon (big sky)", "high horizon (big ground)",
    "vignette focus", "leading lines",
    "light from top-left", "light from top-right", "light from above",
]

def _sanitize_filename(name):
    name = re.sub(r'[^\w\-. ()]+', '_', name.strip())
    if not name.lower().endswith('.safetensors'):
        name = (name.rsplit('.', 1)[0] if '.' in name else name) + '.safetensors'
    return name

def _convert_url(url):
    url = url.strip()
    if 'huggingface.co' in url and '/blob/' in url: url = url.replace('/blob/', '/resolve/')
    m = re.match(r'https?://civitai\.com/models/\d+(?:/[^?]*)?(?:\?modelVersionId=(\d+))?', url)
    if m and m.group(1): url = f"https://civitai.com/api/download/models/{m.group(1)}"
    elif m: return None, "❌ Civitai: need download link."
    return url, None

def download_lora_from_url(url, custom_name, is_civitai=False):
    if not url or not url.strip(): return "❌ Paste a URL.", *([gr.update()] * 3)
    direct_url, error = _convert_url(url.strip())
    if error: return error, *([gr.update()] * 3)
    if is_civitai: direct_url = _append_civitai_token(direct_url)
    guessed = os.path.basename(urllib.parse.unquote(urllib.parse.urlparse(direct_url.split("?")[0]).path))
    filename = _sanitize_filename(custom_name) if custom_name and custom_name.strip() else (
        _sanitize_filename(guessed) if guessed and '.' in guessed else f"lora_{int(time.time())}.safetensors")
    dest = os.path.join(LORA_DIR, filename)
    if os.path.exists(dest): return f"⚠️ Exists: {filename}", *([gr.update(choices=scan_loras())] * 3)
    safe_url = direct_url.replace('"', '\\"')
    ret = os.system(f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{safe_url}" -d "{LORA_DIR}" -o "{filename}"')
    if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < 1024:
        if os.path.exists(dest): os.remove(dest)
        os.system(f'wget -c -q --show-progress -O "{dest}" "{safe_url}"')
    if not os.path.exists(dest) or os.path.getsize(dest) < 1024:
        if os.path.exists(dest): os.remove(dest)
        return "❌ Failed.", *([gr.update()] * 3)
    return f"✅ {filename} ({os.path.getsize(dest)/(1024**2):.0f} MB)", *([gr.update(choices=scan_loras())] * 3)

def delete_downloaded_lora(name):
    if not name or name == "(no LoRAs found)": return "❌ None.", *([gr.update()] * 3)
    p = os.path.join(LORA_DIR, name)
    if os.path.exists(p) and not os.path.islink(p):
        os.remove(p); return f"✅ Deleted: {name}", *([gr.update(choices=scan_loras(), value=None)] * 3)
    return f"ℹ️ Can't delete.", *([gr.update()] * 3)

def scan_loras():
    files = set()
    if os.path.exists(LORA_FOLDER):
        for f in glob.glob(os.path.join(LORA_FOLDER, "*.safetensors")): files.add(os.path.basename(f))
    for f in glob.glob(os.path.join(LORA_DIR, "*.safetensors")): files.add(os.path.basename(f))
    return sorted(files) or ["(no LoRAs found)"]

def refresh_lora_list():
    return [gr.update(choices=scan_loras(), value=None)] * 3

def _gen_wrap(pos, neg, w, h, bs, seed, steps, cfg, samp, sched, den, enh,
              dpo, dpo_s,
              zs_en, zs_int, zs_ibias):
    yield from generate(pos, neg, w, h, bs, seed, steps, cfg, samp, sched, den,
                        enhanced=enh, flow_dpo_enabled=dpo, flow_dpo_strength=dpo_s,
                        zsampler_enabled=zs_en, zs_intensity=zs_int, zs_ibias=zs_ibias)

def _dualcfg_wrap(pos, neg, w, h, bs, seed, total_steps, cfg_low, cfg_high,
                  split_step, samp, sched, dpo, dpo_s):
    yield from _generate_dual_cfg(pos, neg, w, h, bs, seed, total_steps,
                                   cfg_low, cfg_high, split_step, samp, sched,
                                   flow_dpo_enabled=dpo, flow_dpo_strength=dpo_s,
)

def _enh_btn(t): return enhance_prompt(t, auto_unload=True)
def _desc_btn(i): return describe_image(i, auto_unload=True)
def _copy_btn(t): return t

def _build_slider_html(before_pil, after_pil):
    max_dim = 1024; w, h = after_pil.size
    if max(w, h) > max_dim:
        s = max_dim / max(w, h)
        before_pil = before_pil.resize((int(w*s), int(h*s)), Image.LANCZOS)
        after_pil = after_pil.resize((int(w*s), int(h*s)), Image.LANCZOS); w, h = after_pil.size
    def to_b64(img):
        buf = io.BytesIO(); img.save(buf, format="JPEG", quality=90)
        return base64.b64encode(buf.getvalue()).decode()
    b64_b = to_b64(before_pil); b64_a = to_b64(after_pil)
    uid = f"cmp{int(time.time()*1000)}"
    return f"""
<style>
.{uid}-wrap{{position:relative;width:100%;max-width:{w}px;aspect-ratio:{w}/{h};margin:0 auto;overflow:hidden;border-radius:12px;box-shadow:0 4px 20px rgba(0,0,0,0.2)}}
.{uid}-wrap img{{position:absolute;top:0;left:0;width:100%;height:100%;object-fit:cover;display:block;pointer-events:none}}
.{uid}-before{{position:absolute;inset:0;z-index:1;clip-path:inset(0 50% 0 0)}}
.{uid}-after{{position:absolute;inset:0;z-index:0}}
.{uid}-label{{position:absolute;top:10px;padding:4px 12px;background:rgba(0,0,0,0.55);backdrop-filter:blur(6px);color:#fff;font-size:12px;font-weight:600;border-radius:20px;z-index:5;pointer-events:none}}
.{uid}-lb{{left:10px}}.{uid}-la{{right:10px}}
.{uid}-line{{position:absolute;top:0;bottom:0;left:50%;width:3px;transform:translateX(-50%);z-index:3;pointer-events:none;background:linear-gradient(to bottom,transparent 0%,rgba(255,255,255,0.4) 15%,rgba(255,255,255,0.9) 40%,rgba(255,255,255,0.9) 60%,rgba(255,255,255,0.4) 85%,transparent 100%)}}
.{uid}-knob{{position:absolute;top:50%;left:50%;width:44px;height:44px;transform:translate(-50%,-50%);background:white;border-radius:50%;z-index:4;display:flex;align-items:center;justify-content:center;box-shadow:0 4px 16px rgba(0,0,0,0.2);font-size:20px;color:#555;pointer-events:none}}
.{uid}-range{{position:absolute;top:0;left:0;width:100%;height:100%;z-index:10;opacity:0;cursor:grab;margin:0;padding:0;-webkit-appearance:none;appearance:none;background:transparent}}
.{uid}-range::-webkit-slider-thumb{{-webkit-appearance:none;appearance:none;width:44px;height:100vh;cursor:grab;background:transparent}}
.{uid}-range::-moz-range-thumb{{width:44px;height:100%;cursor:grab;background:transparent;border:none}}
</style>
<div class="{uid}-wrap">
<div class="{uid}-after"><img src="data:image/jpeg;base64,{b64_a}" alt="After"/></div>
<div class="{uid}-before" id="{uid}_bf"><img src="data:image/jpeg;base64,{b64_b}" alt="Before"/></div>
<span class="{uid}-label {uid}-lb">Before</span><span class="{uid}-label {uid}-la">After</span>
<div class="{uid}-line" id="{uid}_ln"></div><div class="{uid}-knob" id="{uid}_kb">⇔</div>
<input type="range" min="0" max="100" value="50" class="{uid}-range" id="{uid}_rng"
oninput="var v=this.value;document.getElementById('{uid}_bf').style.clipPath='inset(0 '+(100-v)+'% 0 0)';document.getElementById('{uid}_ln').style.left=v+'%';document.getElementById('{uid}_kb').style.left=v+'%';"/>
</div>"""

def _upscale_wrapper(pil_image, resolution, color_correction, enable_debug,
                     pre_downscale, input_noise, latent_noise,
                     refine_enabled, refine_prompt, refine_negative,
                     refine_denoise, refine_steps, refine_enhanced,
                     refine_lora_name, refine_lora_strength,
                     auto_color_enabled, auto_color_strength, auto_color_protect_skin,
                     final_upscale_enabled, final_resolution,
                     dd_enabled, dd_amount, dd_start, dd_end,
                     dd_bias, dd_exponent, dd_start_offset, dd_end_offset,
                     dd_fade, dd_smooth, dd_cfg_override,
                     dd_denoise, dd_steps):
    last_img = None; saved_path = None
    for img, status_text in upscale_image(
            pil_image, resolution, color_correction, enable_debug,
            pre_downscale, input_noise, latent_noise,
            refine_enabled, refine_prompt, refine_negative,
            refine_denoise, refine_steps, refine_enhanced,
            refine_lora_name, refine_lora_strength,
            auto_color_enabled, auto_color_strength, auto_color_protect_skin,
            final_upscale_enabled, final_resolution,
            dd_enabled=dd_enabled, dd_amount=dd_amount, dd_start=dd_start, dd_end=dd_end,
            dd_bias=dd_bias, dd_exponent=dd_exponent,
            dd_start_offset=dd_start_offset, dd_end_offset=dd_end_offset,
            dd_fade=dd_fade, dd_smooth=dd_smooth, dd_cfg_override=dd_cfg_override,
            dd_denoise=dd_denoise, dd_steps=dd_steps):
        if img is not None: last_img = img
        if "✅" in status_text and last_img is not None:
            ts = int(time.time()); saved_path = f"{OUTPUT_DIR}/upscaled_{ts}.png"; last_img.save(saved_path)
            before, after = _get_comparison_images()
            if before is not None and after is not None: html = _build_slider_html(before, after)
            else:
                buf = io.BytesIO(); last_img.save(buf, format="JPEG", quality=90)
                html = f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}" style="max-width:100%;border-radius:8px;">'
            yield html, status_text, saved_path
        else:
            yield f"<p style='color:#aaa;text-align:center;font-size:14px;'>{status_text}</p>", status_text, None

import json as _json

with gr.Blocks(title="Z-Image Turbo") as demo:
    gr.Markdown("## Z-Image Turbo — ComfyUI + Gradio")
    with gr.Tabs():

        # ── 🎨 GENERATE ──
        with gr.TabItem("🎨 Generate"):
            with gr.Row():
                with gr.Column(scale=1):
                    gallery = gr.Gallery(label="Output", columns=2, height=500, preview=False)
                    with gr.Row():
                        status = gr.Textbox(label="Status", interactive=False, value="Ready", scale=3)
                        used_seed = gr.Number(label="Seed", interactive=False, scale=1)
                with gr.Column(scale=1):
                    positive_prompt = gr.Textbox(label="Positive prompt", lines=3, placeholder="a cinematic portrait of...")
                    negative_prompt = gr.Textbox(label="Negative prompt", lines=1, value="blurry ugly bad,")
                    with gr.Row():
                        run_btn = gr.Button("🎨 Generate", variant="primary", scale=3)
                        stop_btn = gr.Button("⏹️ Stop", variant="stop", scale=1)
                    with gr.Accordion("⚙️ Settings", open=False):
                        with gr.Row():
                            width = gr.Slider(256, 2048, value=1024, step=64, label="W")
                            height = gr.Slider(256, 2048, value=1024, step=64, label="H")
                        with gr.Row():
                            steps = gr.Slider(1, 50, value=9, step=1, label="Steps")
                            cfg = gr.Slider(0.5, 15.0, value=1.0, step=0.1, label="CFG")
                        with gr.Row():
                            sampler_name = gr.Dropdown(SAMPLERS, value="euler", label="Sampler")
                            scheduler = gr.Dropdown(SCHEDULERS, value="simple", label="Scheduler")
                        with gr.Row():
                            denoise = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label="Denoise")
                            batch_size = gr.Slider(1, 8, value=1, step=1, label="Batch")
                        seed = gr.Number(value=0, label="Seed (0 = random)", precision=0)
                    enhanced = gr.Checkbox(label="✨ Enhanced Texture", value=False)
                    with gr.Accordion("🚀 Z-Sampler Turbo (BRAVO)", open=False):
                        zsampler_enabled = gr.Checkbox(label="🚀 Enable Z-Sampler Turbo", value=False,
                            info="3-stage BRAVO pipeline. Overrides Standard/Enhanced.")
                        zs_intensity = gr.Slider(0.0, 1.0, value=0.5, step=0.1, label="Intensity",
                            info="0.0 = soft, 0.5 = default, 1.0 = vivid")
                        zs_ibias = gr.Slider(-1.0, 1.0, value=0.0, step=0.2, label="Bias",
                            info="Keep at 0.0. Adjust if too dark/bright.")
                    with gr.Accordion("🔆 Flow-DPO Lighting", open=False):
                        flow_dpo_enabled = gr.Checkbox(label="🔆 Enable", value=False)
                        flow_dpo_strength = gr.Slider(0.3, 1.5, value=0.8, step=0.05, label="Strength")

            gen_event = run_btn.click(fn=_gen_wrap,
                inputs=[positive_prompt, negative_prompt, width, height, batch_size, seed,
                        steps, cfg, sampler_name, scheduler, denoise, enhanced,
                        flow_dpo_enabled, flow_dpo_strength,
                        zsampler_enabled, zs_intensity, zs_ibias],
                outputs=[gallery, used_seed, status])
            stop_btn.click(fn=request_stop, inputs=[], outputs=[status], cancels=[gen_event])

        # ── 🧠 MODEL ──
        with gr.TabItem("🧠 Model"):
            gr.Markdown("### Model Manager — download & switch")
            model_choices = scan_diffusion_models()
            with gr.Accordion("⬇️ Download New Model", open=False):
                mdl_url = gr.Textbox(label="URL", lines=1)
                mdl_custom_name = gr.Textbox(label="Filename (optional)", lines=1)
                mdl_civitai = gr.Checkbox(label="Civitai (append API token)", value=False)
                mdl_dl_btn = gr.Button("⬇️ Download", variant="primary")
                mdl_dl_status = gr.Textbox(label="Status", lines=2, interactive=False)
            mdl_select = gr.Dropdown(choices=model_choices, value=current_model_name, label="Model")
            mdl_dtype = gr.Dropdown(choices=WEIGHT_DTYPES, value="fp8_e4m3fn_fast", label="Weight type")
            with gr.Row():
                mdl_load_btn = gr.Button("🔄 Load", variant="primary", scale=3)
                mdl_refresh_btn = gr.Button("🔄", variant="secondary", scale=1)
                mdl_delete_btn = gr.Button("🗑️", variant="secondary", scale=1)
            mdl_status = gr.Textbox(label="Status", lines=3, interactive=False, value=f"✅ Active: {current_model_name}")
            mdl_dl_btn.click(fn=download_model, inputs=[mdl_url, mdl_custom_name, mdl_civitai], outputs=[mdl_dl_status, mdl_select])
            mdl_load_btn.click(fn=switch_model, inputs=[mdl_select, mdl_dtype], outputs=[mdl_status])
            mdl_refresh_btn.click(fn=lambda: gr.update(choices=scan_diffusion_models()), inputs=[], outputs=[mdl_select])
            mdl_delete_btn.click(fn=delete_model, inputs=[mdl_select], outputs=[mdl_status, mdl_select])

        # ── 🔧 LORA ──
        with gr.TabItem("🔧 LoRA"):
            gr.Markdown(f"### LoRA Manager (⚡ Bypass mode — fp8-safe, MODEL only)\n📁 `{LORA_FOLDER}`")
            with gr.Accordion("🔗 Download LoRA from URL", open=False):
                dl_url = gr.Textbox(label="URL", lines=1)
                dl_custom_name = gr.Textbox(label="Filename (optional)", lines=1)
                dl_civitai = gr.Checkbox(label="Civitai (append API token)", value=False)
                with gr.Row():
                    dl_btn = gr.Button("⬇️ Download", variant="primary", scale=3)
                    dl_del_btn = gr.Button("🗑️ Delete", variant="secondary", scale=1)
                dl_status = gr.Textbox(label="Status", lines=2, interactive=False)
            refresh_btn = gr.Button("🔄 Refresh", variant="secondary")
            lora_choices = scan_loras()
            with gr.Group():
                gr.Markdown("**LoRA 1**")
                with gr.Row():
                    enable1 = gr.Checkbox(label="On", value=False, min_width=50)
                    select1 = gr.Dropdown(choices=lora_choices, label="LoRA", scale=4)
                mstr1 = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="Strength")
            with gr.Group():
                gr.Markdown("**LoRA 2**")
                with gr.Row():
                    enable2 = gr.Checkbox(label="On", value=False, min_width=50)
                    select2 = gr.Dropdown(choices=lora_choices, label="LoRA", scale=4)
                mstr2 = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="Strength")
            with gr.Group():
                gr.Markdown("**LoRA 3**")
                with gr.Row():
                    enable3 = gr.Checkbox(label="On", value=False, min_width=50)
                    select3 = gr.Dropdown(choices=lora_choices, label="LoRA", scale=4)
                mstr3 = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="Strength")
            with gr.Accordion("⚙️ LoRA Settings", open=False):
                auto_scale = gr.Checkbox(label="⚖️ Auto-scale", value=True)
                apply_clip = gr.Checkbox(label="🔓 Also apply to CLIP (not recommended)", value=False)
                clip_ratio = gr.Slider(0.1, 1.0, value=0.3, step=0.05, label="CLIP ratio")
            with gr.Row():
                load_btn = gr.Button("✅ Load LoRAs", variant="primary")
                unload_btn = gr.Button("🗑️ Unload", variant="secondary")
            lora_status = gr.Textbox(label="Status", lines=8, interactive=False, value="ℹ️ No LoRAs loaded.")
            with gr.Accordion("🔬 A/B Test — verify a LoRA actually works", open=False):
                gr.Markdown("Generates the same seed **with** and **without** the LoRA. If the images are identical, the LoRA isn't working.")
                ab_lora = gr.Dropdown(choices=lora_choices, label="LoRA to test")
                with gr.Row():
                    ab_strength = gr.Slider(0.1, 1.5, value=1.0, step=0.05, label="Strength")
                    ab_steps = gr.Slider(4, 12, value=8, step=1, label="Steps")
                ab_prompt = gr.Textbox(label="Test prompt (optional)", lines=1,
                    placeholder="Leave empty for a generic prompt — or use the LoRA's trigger word")
                ab_seed = gr.Number(value=0, label="Seed (0 = random)", precision=0)
                ab_btn = gr.Button("🔬 Run A/B Test", variant="primary")
                ab_gallery = gr.Gallery(label="A = no LoRA  |  B = with LoRA", columns=2, height=400, preview=False)
                ab_status = gr.Textbox(label="Result", interactive=False)
            dl_btn.click(fn=download_lora_from_url, inputs=[dl_url, dl_custom_name, dl_civitai], outputs=[dl_status, select1, select2, select3])
            dl_del_btn.click(fn=delete_downloaded_lora, inputs=[select1], outputs=[dl_status, select1, select2, select3])
            refresh_btn.click(fn=refresh_lora_list, inputs=[], outputs=[select1, select2, select3])
            load_btn.click(fn=load_loras, inputs=[select1, mstr1, enable1, select2, mstr2, enable2, select3, mstr3, enable3, auto_scale, apply_clip, clip_ratio], outputs=[lora_status])
            unload_btn.click(fn=unload_all_loras, inputs=[], outputs=[lora_status])
            ab_btn.click(fn=lora_ab_test, inputs=[ab_lora, ab_strength, ab_prompt, ab_seed, ab_steps], outputs=[ab_gallery, ab_status])
            refresh_btn.click(fn=lambda: gr.update(choices=scan_loras()), inputs=[], outputs=[ab_lora])

        # ── ⬆️ UPSCALE ──
        with gr.TabItem("⬆️ Upscale"):
            gr.Markdown("### SeedVR2 → Refine → Detail Daemon → Auto Color")
            with gr.Row():
                with gr.Column(scale=1):
                    up_input = gr.Image(type="pil", label="Image", height=400)
                    use_last_btn = gr.Button("🔄 Use last generated", variant="secondary")
                    up_resolution = gr.Slider(720, 4096, value=2048, step=64, label="Resolution")
                    up_color = gr.Dropdown(COLOR_METHODS, value="wavelet", label="Color correction")
                    with gr.Accordion("🎛️ Artifact Control", open=False):
                        up_input_noise = gr.Slider(0.0, 0.5, value=0.15, step=0.05, label="Input noise")
                        up_latent_noise = gr.Slider(0.0, 0.3, value=0.05, step=0.05, label="Latent noise")
                        up_pre_downscale = gr.Slider(0.5, 1.0, value=1.0, step=0.05, label="Pre-downscale")
                    with gr.Accordion("🔧 Stage 2 — Z-Image Refine", open=False):
                        refine_enabled = gr.Checkbox(label="🔧 Enable refine", value=False)
                        refine_prompt = gr.Textbox(label="Prompt", lines=2, value="high quality, detailed, sharp")
                        refine_negative = gr.Textbox(label="Negative", lines=1, value="blurry ugly bad")
                        refine_denoise = gr.Slider(0.1, 0.6, value=0.3, step=0.05, label="Denoise")
                        refine_steps = gr.Slider(3, 12, value=6, step=1, label="Steps")
                        refine_enhanced = gr.Checkbox(label="✨ Enhanced two-pass", value=True)
                        gr.Markdown("**Refine LoRA** (optional)")
                        refine_lora_name = gr.Dropdown(choices=["(none)"] + scan_loras(), value="(none)", label="LoRA for refine pass")
                        refine_lora_strength = gr.Slider(0.3, 1.5, value=0.8, step=0.05, label="Refine LoRA strength")
                        refine_lora_refresh = gr.Button("🔄 Refresh LoRA list", variant="secondary", size="sm")
                        refine_lora_refresh.click(fn=lambda: gr.update(choices=["(none)"] + scan_loras()), inputs=[], outputs=[refine_lora_name])
                    with gr.Accordion("🎨 Auto Color (FameGrid)", open=False):
                        auto_color_enabled = gr.Checkbox(label="🎨 Enable auto color correction", value=False)
                        auto_color_strength = gr.Slider(0.5, 2.0, value=1.10, step=0.05, label="Strength")
                        auto_color_protect_skin = gr.Checkbox(label="Protect skin tones", value=True)
                    with gr.Accordion("🔎 Final Upscale (Lanczos)", open=False):
                        final_upscale_enabled = gr.Checkbox(label="🔎 Enable final Lanczos upscale", value=False,
                            info="Simple Lanczos resize after all processing — good for 4K output")
                        final_resolution = gr.Slider(2048, 4096, value=3072, step=64, label="Final resolution (longest side)")
                    with gr.Accordion("🔍 Detail Daemon Sampler", open=False):
                        dd_enabled = gr.Checkbox(label="🔍 Enable Detail Daemon", value=False,
                            info="Post-refine pass — adjusts sigma per step for enhanced detail")
                        dd_amount = gr.Slider(-1.0, 2.0, value=0.1, step=0.01, label="Detail Amount",
                            info="Main control. Z-Image: 0.1–1.0. Negative = reduce detail.")
                        with gr.Row():
                            dd_start = gr.Slider(0.0, 1.0, value=0.2, step=0.01, label="Start")
                            dd_end = gr.Slider(0.0, 1.0, value=0.8, step=0.01, label="End")
                        with gr.Row():
                            dd_bias = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Bias",
                                info="Shifts peak forward/back")
                            dd_exponent = gr.Slider(0.0, 5.0, value=1.0, step=0.05, label="Exponent",
                                info="Curve shape. 0=flat, 1=smooth")
                        with gr.Accordion("⚙️ DD Advanced", open=False):
                            with gr.Row():
                                dd_start_offset = gr.Slider(-1.0, 1.0, value=0.0, step=0.01, label="Start Offset")
                                dd_end_offset = gr.Slider(-1.0, 1.0, value=0.0, step=0.01, label="End Offset")
                            with gr.Row():
                                dd_fade = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Fade")
                                dd_smooth = gr.Checkbox(label="Smooth curve", value=True)
                            dd_cfg_override = gr.Slider(0.0, 5.0, value=1.0, step=0.1, label="CFG Scale",
                                info="0 = auto-detect from model")
                        with gr.Row():
                            dd_denoise = gr.Slider(0.1, 0.5, value=0.25, step=0.05, label="Denoise")
                            dd_steps = gr.Slider(3, 12, value=6, step=1, label="Steps")
                    up_debug = gr.Checkbox(label="Debug", value=False)
                    up_btn = gr.Button("⬆️ Upscale", variant="primary")
                with gr.Column(scale=1):
                    up_status = gr.Textbox(label="Status", interactive=False, value="Ready")
                    up_result = gr.HTML(value="<p style='color:#888;text-align:center;'>Upload an image and click Upscale</p>")
                    up_download = gr.File(label="📥 Download upscaled image", visible=True, interactive=False)
            use_last_btn.click(fn=_get_last_generated, inputs=[], outputs=up_input)
            up_btn.click(fn=_upscale_wrapper,
                inputs=[up_input, up_resolution, up_color, up_debug, up_pre_downscale, up_input_noise, up_latent_noise,
                        refine_enabled, refine_prompt, refine_negative, refine_denoise, refine_steps, refine_enhanced,
                        refine_lora_name, refine_lora_strength, auto_color_enabled, auto_color_strength, auto_color_protect_skin,
                        final_upscale_enabled, final_resolution,
                        dd_enabled, dd_amount, dd_start, dd_end,
                        dd_bias, dd_exponent, dd_start_offset, dd_end_offset,
                        dd_fade, dd_smooth, dd_cfg_override,
                        dd_denoise, dd_steps],
                outputs=[up_result, up_status, up_download])

        # ── 🩹 FIX & INPAINT ──
        with gr.TabItem("🩹 Fix"):
            gr.Markdown("### Auto-Fix & Manual Inpaint")
            with gr.Tabs():
                with gr.TabItem("🤖 Auto-Fix"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            af_input = gr.Image(type="pil", label="Image", height=400)
                            af_use_last = gr.Button("🔄 Use last generated", variant="secondary")
                            with gr.Row():
                                af_faces = gr.Checkbox(label="👤 Faces", value=True)
                                af_hands = gr.Checkbox(label="🤚 Hands", value=True)
                                af_bodies = gr.Checkbox(label="🧍 Bodies", value=False)
                            af_prompt = gr.Textbox(label="Prompt (blank = auto)", lines=2, value="")
                            af_negative = gr.Textbox(label="Negative", lines=1, value="blurry ugly bad deformed extra fingers missing fingers")
                            af_denoise = gr.Slider(0.1, 1.0, value=0.25, step=0.05, label="Denoise",
                                info="Low (0.2–0.3) = detail only. High (0.5+) = regenerate.")
                            with gr.Accordion("⚙️ Advanced", open=False):
                                af_confidence = gr.Slider(0.1, 0.9, value=0.3, step=0.05, label="Confidence")
                                af_padding = gr.Slider(0.1, 0.6, value=0.3, step=0.05, label="Padding")
                                af_resolution = gr.Slider(512, 2048, value=1024, step=64, label="Crop resolution",
                                    info="Higher = more detail. Detected region upscaled to this before sampling.")
                            with gr.Accordion("⬆️ SeedVR2 Detail (on crop)", open=False):
                                af_seedvr2 = gr.Checkbox(label="⬆️ Upscale inpainted region with SeedVR2", value=False,
                                    info="Runs SeedVR2 on the cropped result before stitching — adds detail")
                                af_seedvr2_res = gr.Slider(1024, 2048, value=1536, step=64, label="SeedVR2 target resolution")
                            af_btn = gr.Button("🤖 Auto-Fix", variant="primary")
                        with gr.Column(scale=1):
                            af_status = gr.Textbox(label="Status", interactive=False, value="Ready")
                            af_output = gr.Image(type="pil", label="Result", height=500)
                    af_use_last.click(fn=_get_last_generated, inputs=[], outputs=af_input)
                    af_btn.click(fn=auto_fix_image, inputs=[af_input, af_faces, af_hands, af_bodies, af_prompt, af_negative, af_denoise, af_confidence, af_padding, af_resolution, af_seedvr2, af_seedvr2_res], outputs=[af_output, af_status])
                with gr.TabItem("🖌️ Inpaint"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            ip_editor = gr.ImageEditor(type="pil", label="Paint mask", height=450, brush=gr.Brush(colors=["#FFFFFF"], default_size=30), eraser=gr.Eraser(default_size=20), layers=True)
                            ip_load_last = gr.Button("🔄 Load last generated", variant="secondary")
                            ip_prompt = gr.Textbox(label="What should be here?", lines=2, placeholder="e.g. detailed hand with 5 fingers")
                            ip_negative = gr.Textbox(label="Negative", lines=1, value="blurry ugly bad")
                            with gr.Row():
                                ip_denoise = gr.Slider(0.1, 1.0, value=0.7, step=0.05, label="Denoise",
                                    info="Low (0.2–0.3) = add detail only. High (0.7+) = regenerate content.")
                                ip_steps = gr.Slider(4, 15, value=9, step=1, label="Steps")
                            ip_resolution = gr.Slider(512, 2048, value=1024, step=64, label="Crop resolution",
                                info="Higher = more detail. Masked area is upscaled to this before sampling.")
                            with gr.Accordion("⬆️ SeedVR2 Detail (on crop)", open=False):
                                ip_seedvr2 = gr.Checkbox(label="⬆️ Upscale inpainted region with SeedVR2", value=False,
                                    info="Runs SeedVR2 on the cropped result before stitching")
                                ip_seedvr2_res = gr.Slider(1024, 2048, value=1536, step=64, label="SeedVR2 target resolution")
                            ip_btn = gr.Button("🖌️ Inpaint", variant="primary")
                        with gr.Column(scale=1):
                            ip_status = gr.Textbox(label="Status", interactive=False, value="Ready")
                            ip_output = gr.Image(type="pil", label="Result", height=500)
                    ip_load_last.click(fn=_get_last_generated, inputs=[], outputs=ip_editor)
                    ip_btn.click(fn=manual_inpaint, inputs=[ip_editor, ip_prompt, ip_negative, ip_denoise, ip_steps, ip_resolution, ip_seedvr2, ip_seedvr2_res], outputs=[ip_output, ip_status])
                with gr.TabItem("🦴 Pose Edit"):
                    gr.Markdown("**Edit pose by dragging joints, then transfer with Qwen Edit.**\n"
                                "1\ufe0f\u20e3 Upload \u2192 Extract \u2192 **download** the editor HTML\n"
                                "2\ufe0f\u20e3 Open it in your browser, drag joints, click **Copy Pose JSON**\n"
                                "3\ufe0f\u20e3 Paste JSON here \u2192 Apply \u2192 Generate")
                    with gr.Row():
                        with gr.Column(scale=1):
                            pe_input = gr.Image(type="pil", label="Upload image", height=250)
                            pe_extract_btn = gr.Button("\U0001f9b4 Extract Pose", variant="primary")
                            pe_download = gr.File(label="\U0001f4e5 Download pose editor (open in browser)", interactive=False)
                            gr.Markdown("##### Paste edited JSON:")
                            pe_json_box = gr.Textbox(label="Pose JSON (from editor)", lines=3,
                                placeholder="Drag joints in the editor, click Copy Pose JSON, paste here")
                            pe_apply_btn = gr.Button("\U0001f4cc Apply pasted pose", variant="secondary")
                        with gr.Column(scale=1):
                            pe_status = gr.Textbox(label="Status", interactive=False, value="Ready", lines=3)
                            pe_skel_preview = gr.Image(type="pil", label="Skeleton preview", height=280, interactive=False)
                            qe_prompt = gr.Textbox(label="Extra prompt (optional)", lines=1,
                                placeholder="e.g. photo taken in a studio, natural lighting")
                            qe_negative = gr.Textbox(label="Negative", lines=1,
                                value="blurry, deformed, extra limbs, different person, changed clothes")
                            with gr.Row():
                                qe_lightning = gr.Checkbox(label="\u26a1 Lightning 4-step", value=True)
                                qe_steps = gr.Slider(4, 20, value=8, step=1, label="Steps (ignored if Lightning)")
                            with gr.Row():
                                qe_cfg = gr.Slider(1.0, 4.0, value=1.0, step=0.1, label="CFG", info="1.0 with Lightning")
                                qe_mp = gr.Slider(0.5, 1.5, value=1.0, step=0.1, label="Megapixels")
                            with gr.Row():
                                qe_seed = gr.Number(value=0, label="Seed (0=random)", precision=0)
                                qe_unload = gr.Checkbox(label="Unload after (free VRAM)", value=False)
                            qe_btn = gr.Button("\U0001f3ad Transfer pose with Qwen Edit", variant="primary")
                            pe_result = gr.Image(type="pil", label="Result", height=300)
                    pe_orig = gr.State(value=None)
                    pe_skel = gr.State(value=None)
                    pe_pose_dict = gr.State(value=None)
                    def _pe_extract(img):
                        orig, overlay, skel, pdict, status = extract_pose(img)
                        if orig is None: return None, None, None, status, None, overlay
                        html_path = _save_pose_editor_html(orig, pdict)
                        return orig, skel, pdict, status, html_path, overlay
                    pe_extract_btn.click(fn=_pe_extract, inputs=[pe_input],
                        outputs=[pe_orig, pe_skel, pe_pose_dict, pe_status, pe_download, pe_skel_preview])
                    pe_apply_btn.click(fn=apply_edited_pose,
                        inputs=[pe_orig, pe_pose_dict, pe_json_box],
                        outputs=[pe_skel, pe_skel_preview, pe_pose_dict, pe_status])
                    qe_btn.click(fn=qwen_pose_transfer,
                        inputs=[pe_orig, pe_skel, qe_prompt, qe_negative, qe_steps, qe_cfg, qe_seed,
                                qe_lightning, qe_mp, qe_unload],
                        outputs=[pe_result, pe_status])

        # ── 🎲 DUAL CFG ──
        with gr.TabItem("🎲 Dual CFG"):
            gr.Markdown("### Dual CFG — Split-Step Variation\n"
                        "Pass 1 runs low CFG for creative variation, Pass 2 runs higher CFG for prompt adherence. "
                        "Works best with simple prompts. May fail on complex ones.")
            with gr.Row():
                with gr.Column(scale=1):
                    dc_gallery = gr.Gallery(label="Output", columns=3, height=500, preview=False)
                    with gr.Row():
                        dc_status = gr.Textbox(label="Status", interactive=False, value="Ready", scale=3)
                        dc_seed_out = gr.Number(label="Seed", interactive=False, scale=1)
                with gr.Column(scale=1):
                    dc_pos = gr.Textbox(label="Positive prompt", lines=3, placeholder="Beautiful woman riding a bike")
                    dc_neg = gr.Textbox(label="Negative prompt", lines=1, value="")
                    with gr.Row():
                        dc_run = gr.Button("🎲 Generate", variant="primary", scale=3)
                        dc_stop = gr.Button("⏹️ Stop", variant="stop", scale=1)
                    with gr.Row():
                        dc_cfg_low = gr.Slider(0.05, 1.0, value=0.4, step=0.05, label="CFG Low (Pass 1)",
                            info="0.1–0.5 recommended")
                        dc_cfg_high = gr.Slider(0.5, 5.0, value=1.0, step=0.1, label="CFG High (Pass 2)",
                            info="1.0+ recommended")
                    with gr.Row():
                        dc_split = gr.Slider(1, 5, value=2, step=1, label="Split at step",
                            info="How many steps use low CFG")
                        dc_steps = gr.Slider(4, 30, value=10, step=1, label="Total steps")
                    with gr.Row():
                        dc_width = gr.Slider(256, 2048, value=1024, step=64, label="W")
                        dc_height = gr.Slider(256, 2048, value=1024, step=64, label="H")
                    with gr.Row():
                        dc_batch = gr.Slider(1, 9, value=9, step=1, label="Batch",
                            info="Original workflow uses 9")
                        dc_seed = gr.Number(value=0, label="Seed (0 = random)", precision=0)
                    with gr.Accordion("⚙️ Advanced", open=False):
                        dc_sampler = gr.Dropdown(SAMPLERS, value="euler", label="Sampler")
                        dc_sched = gr.Dropdown(SCHEDULERS, value="simple", label="Scheduler")
                        dc_dpo = gr.Checkbox(label="🔆 Flow-DPO", value=False)
                        dc_dpo_s = gr.Slider(0.3, 1.5, value=0.8, step=0.05, label="DPO Strength")

            dc_event = dc_run.click(fn=_dualcfg_wrap,
                inputs=[dc_pos, dc_neg, dc_width, dc_height, dc_batch, dc_seed,
                        dc_steps, dc_cfg_low, dc_cfg_high, dc_split,
                        dc_sampler, dc_sched, dc_dpo, dc_dpo_s, ],
                outputs=[dc_gallery, dc_seed_out, dc_status])
            dc_stop.click(fn=request_stop, inputs=[], outputs=[dc_status], cancels=[dc_event])

        # ── ✍️ PROMPT ENHANCE ──
        with gr.TabItem("✍️ Prompt"):
            gr.Markdown("### QwenVL Prompt Enhancement")
            with gr.Tabs():
                with gr.TabItem("📝 Enhance"):
                    pe_text_input = gr.Textbox(label="Idea", lines=2, placeholder="mermaid at sunset")
                    pe_enhance_btn = gr.Button("✍️ Enhance", variant="primary")
                    pe_text_output = gr.Textbox(label="Enhanced", lines=6, interactive=True)
                    pe_copy_text_btn = gr.Button("📋 Copy to Generate", variant="secondary")
                    pe_enhance_btn.click(fn=_enh_btn, inputs=[pe_text_input], outputs=[pe_text_output])
                    pe_copy_text_btn.click(fn=_copy_btn, inputs=[pe_text_output], outputs=[positive_prompt])
                with gr.TabItem("🖼️ Describe"):
                    pe_ref = gr.Image(type="pil", label="Reference", height=300)
                    pe_desc_btn = gr.Button("🖼️ Describe", variant="primary")
                    pe_img_out = gr.Textbox(label="Description", lines=6, interactive=True)
                    pe_copy_img = gr.Button("📋 Copy to Generate", variant="secondary")
                    pe_desc_btn.click(fn=_desc_btn, inputs=[pe_ref], outputs=[pe_img_out])
                    pe_copy_img.click(fn=_copy_btn, inputs=[pe_img_out], outputs=[positive_prompt])
            pe_unload = gr.Button("🗑️ Unload QwenVL", variant="secondary")
            pe_stat = gr.Textbox(label="", interactive=False, value="", max_lines=1)
            pe_unload.click(fn=force_unload_qwen, inputs=[], outputs=[pe_stat])

demo.launch(share=True, debug=True)